# Cross-Lingual Memory in LLM Agents: batch01_pilot_r2 (items: batch01_pilot)

Runs the experiment pipeline from `phase2/design_spec.md` on Kaggle **GPU T4 x2** (internet on, secret `HF_TOKEN` attached).

| Step | What | Rough time |
|---|---|---|
| 1-3 | Install vLLM, read HF token, write code + data | ~6 min |
| 4 | Translate fillers + build histories + English translations (Qwen3-8B) | ~10 min |
| 5 | Retrieval: BGE-M3, multilingual-E5, BM25 | ~5 min |
| 6 | Token-length check | ~1 min |
| 7 | Reader: Qwen3-8B | ~20-30 min |
| 8 | Reader: Llama-3.1-8B | ~20-30 min |
| 9 | Judge: Qwen3-14B-AWQ | ~15 min |
| 10 | Report + download zip | ~1 min |

Times are estimates until the first run. Every stage caches its outputs: if the session dies, re-run from the top and finished work is skipped.
For an unattended run use **Save Version -> Save & Run All (Commit)**; outputs then appear under the notebook's *Output* tab.

## 1. Install

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install -q -U vllm sentence-transformers 2>&1 | tail -2
# vLLM upgrades PyTorch; Kaggle's preinstalled torchaudio then no longer matches and breaks
# `import transformers`. We never use audio, so remove it.
!pip uninstall -y -q torchaudio
import subprocess, os
CHECK = ("import torch, vllm; from vllm import LLM; import transformers; "
         "print('torch', torch.__version__, '| cuda', torch.version.cuda, '| GPUs visible:', torch.cuda.device_count(), "
         "'| vllm', vllm.__version__, '| transformers', transformers.__version__); "
         "assert torch.cuda.device_count() == 2, 'PyTorch cannot see both T4 GPUs'")
r = subprocess.run(["python", "-c", CHECK], capture_output=True, text=True)
print(r.stdout.strip() or r.stderr.strip()[-1500:])
if r.returncode != 0:
    raise RuntimeError("STOP: environment check failed (see message above). Copy it to Claude.")
os.environ["CLM_BACKEND"] = "vllm"
print("Environment OK -> backend vllm")

## 2. Hugging Face token (from Kaggle Secrets; never printed)

In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
print("HF token loaded:", bool(os.environ["HF_TOKEN"]))

## 3. Write pipeline code and data

In [ ]:
!mkdir -p /kaggle/working/clm/src/clm /kaggle/working/clm/data/items /kaggle/working/clm/data/haystacks

In [ ]:
%%writefile /kaggle/working/clm/src/clm/__init__.py
"""Cross-Lingual Memory in LLM Agents: experiment pipeline (design_spec.md v1.0)."""


In [ ]:
%%writefile /kaggle/working/clm/src/clm/config.py
"""Fixed experimental settings. Values trace to phase2/design_spec.md (section in brackets)."""
import os
from pathlib import Path

# Data root: the project folder locally, /kaggle/working/clm on Kaggle (set CLM_ROOT).
ROOT = Path(os.environ.get("CLM_ROOT", Path(__file__).resolve().parents[2]))
DATA = ROOT / "data"
RESULTS = ROOT / "results" / os.environ.get("CLM_RUN", "pilot")

LANGS = ("en", "hi", "hg")                       # [2.1]
QUERY_VARIANTS = ("q1", "q2")                    # [1.2]
SETUPS = ("FC", "RM-S1", "RM-S2", "RM-S3", "OR-S1", "OR-S2", "OR-S3")   # [4.1]
TOP_K = 5                                        # [4.2]
RECALL_KS = (1, 3, 5, 10)                        # [4.2]
N_SESSIONS = 10                                  # [3.5] Short haystack
FILLER_POOL_SIZE = 40                            # [3.5]

MODELS = {                                       # [5.1], [4.3], [6.2]
    "qwen3-8b": "Qwen/Qwen3-8B",
    "llama3.1-8b": "meta-llama/Llama-3.1-8B-Instruct",
    "judge": "Qwen/Qwen3-14B-AWQ",
}
TRANSLATOR = "qwen3-8b"                          # [4.3] fixed translator for both readers
EMBEDDER = "BAAI/bge-m3"                         # [4.2]
EMBEDDER_2 = "intfloat/multilingual-e5-large"    # [4.2] robustness, recall only

READER_MAX_TOKENS = 96                           # [5.1]
MAX_MODEL_LEN = 16384                            # [CHECK IN PHASE 4] verified by tokens.py


In [ ]:
%%writefile /kaggle/working/clm/src/clm/generate.py
"""Stage 3 (GPU, one reader per process): answer every query under every setup [design_spec 4.1, 7].

Request id = item|store|query|qv|setup. Outputs are cached in generations_<model>.jsonl; reruns only
fill in missing ids, so a crashed Kaggle session can simply be restarted.
"""
import json

from . import config, haystack, prompts


def memory_text(unit_text, strategy, store, en):
    """How a stored memory is shown to the reader under each strategy. English memories need no translation."""
    if store == "en" or strategy == "S1":
        return unit_text
    if strategy == "S2":
        return en.get(unit_text, unit_text)
    return f"{unit_text} (English: {en.get(unit_text, unit_text)})"   # S3


def build_requests(items, setups=config.SETUPS):
    hist = {(h["item_id"], h["lang"]): h for h in haystack.load_jsonl(config.RESULTS / "histories.jsonl")}
    ret = {}
    if any(s.startswith("RM") for s in setups):
        ret = {(r["item_id"], r["store"], r["query"], r["qv"], r["strategy"]): r["ranked"]
               for r in haystack.load_jsonl(config.RESULTS / "retrieval.jsonl") if r["retriever"] == "bge-m3"}
    en = json.loads((config.RESULTS / "en_translations.json").read_text(encoding="utf-8"))
    reqs = []
    for it in items:
        for s in config.LANGS:
            h = hist[(it["item_id"], s)]
            unit_text = dict(haystack.units(h))
            order = [u for u, _ in haystack.units(h)]
            for q in config.LANGS:
                for qv in config.QUERY_VARIANTS:
                    query = it["text"][q][qv]
                    for setup in setups:
                        rid = f"{it['item_id']}|{s}|{q}|{qv}|{setup}"
                        if setup == "FC":
                            sessions = [[{"role": t["role"], "content": t["content"]} for t in sess]
                                        for sess in h["sessions"]]
                            msgs = prompts.reader_fc(sessions, query)
                        else:
                            kind, strat = setup.split("-")
                            if kind == "RM":
                                chosen = ret[(it["item_id"], s, q, qv, strat)][:config.TOP_K]
                            else:  # oracle: gold (+ distractor) in chronological order
                                chosen = sorted([u for u in (h["gold_unit"], h["distractor_unit"]) if u],
                                                key=order.index)
                            mems = [memory_text(unit_text[u], strat, s, en) for u in chosen]
                            msgs = prompts.reader_memories(mems, query)
                        reqs.append({"id": rid, "messages": msgs})
    return reqs


def run(items_path, model_key, backend=None, setups=config.SETUPS):
    from .llm import Generator
    items = haystack.load_jsonl(items_path)
    reqs = build_requests(items, setups)
    out_path = config.RESULTS / f"generations_{model_key}.jsonl"
    done = {r["id"] for r in haystack.load_jsonl(out_path)} if out_path.exists() else set()
    todo = [r for r in reqs if r["id"] not in done]
    print(f"{model_key}: {len(reqs)} requests, {len(todo)} to run")
    if not todo:
        return
    gen = Generator(model_key, backend=backend)
    chunk = 2000   # write to disk regularly so a crash loses little
    for i in range(0, len(todo), chunk):
        part = todo[i:i + chunk]
        outs = gen.chat([r["messages"] for r in part], max_tokens=config.READER_MAX_TOKENS)
        with open(out_path, "a", encoding="utf-8") as f:
            for r, o in zip(part, outs):
                f.write(json.dumps({"id": r["id"], "model": model_key, "response": o}, ensure_ascii=False) + "\n")
        print(f"  wrote {i + len(part)}/{len(todo)}")


In [ ]:
%%writefile /kaggle/working/clm/src/clm/haystack.py
"""Build conversation histories [design_spec 3.5].

Every item gets a 10-session history per storage language. Filler choice, evidence position and
distractor placement depend only on the item's seed, so the three language versions of a history
differ in language and nothing else. Unanswerable controls reuse their base item's history.
"""
import json
import random

from . import config


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def filler_path(lang):
    """EN pool lives in data/; HI/HG pools come from this run's translate stage, unless frozen copies exist in data/."""
    frozen = config.DATA / "haystacks" / f"filler_pool_{lang}.jsonl"
    return frozen if lang == "en" or frozen.exists() else config.RESULTS / f"filler_pool_{lang}.jsonl"


def load_fillers():
    pools = {}
    for lang in config.LANGS:
        p = filler_path(lang)
        if p.exists():
            pools[lang] = {s["filler_id"]: s for s in load_jsonl(p)}
    return pools


def layout(base):
    """Deterministic layout shared by all languages: which filler goes in which slot, where the distractor is."""
    rng = random.Random(base["filler_seed"])
    filler_ids = [f"FL{i:02d}" for i in rng.sample(range(1, config.FILLER_POOL_SIZE + 1), config.N_SESSIONS - 1)]
    ev = base["evidence_position"]
    slots, it = {}, iter(filler_ids)
    for k in range(1, config.N_SESSIONS + 1):
        slots[k] = "EVIDENCE" if k == ev else next(it)
    host = None
    if base.get("has_distractor"):
        host = rng.choice([k for k in slots if slots[k] != "EVIDENCE" and abs(k - ev) >= 2])
    return slots, host


def build(base, lang, fillers):
    """Return {'sessions': [[turn,...],...], 'gold_unit', 'distractor_unit'}; user turns carry a unit_id."""
    slots, host = layout(base)
    t = base["text"][lang]
    sessions, gold_unit, dis_unit = [], None, None
    for k in range(1, config.N_SESSIONS + 1):
        if slots[k] == "EVIDENCE":
            turns = [{"role": "user", "content": t["memory_turn"]},
                     {"role": "assistant", "content": t["assistant_reply"]},
                     {"role": "user", "content": t["followup_turn"]},
                     {"role": "assistant", "content": t["followup_reply"]}]
        else:
            turns = [dict(x) for x in fillers[lang][slots[k]]["turns"]]
            if k == host:
                turns[0]["content"] = turns[0]["content"].rstrip() + " " + t["distractor_sentence"]
        u = 0
        for turn in turns:
            if turn["role"] == "user":
                u += 1
                turn["unit_id"] = f"s{k}u{u}"
        if slots[k] == "EVIDENCE":
            gold_unit = f"s{k}u1"
        if k == host:
            dis_unit = f"s{k}u1"
        sessions.append(turns)
    return {"sessions": sessions, "gold_unit": gold_unit, "distractor_unit": dis_unit,
            "filler_ids": [slots[k] for k in sorted(slots)]}


def build_all(items, fillers):
    by_id = {i["item_id"]: i for i in items}
    out = []
    for it in items:
        base = it if it.get("is_answerable", True) else by_id[it["base_item"]]
        for lang in config.LANGS:
            h = build(base, lang, fillers)
            out.append({"item_id": it["item_id"], "lang": lang, **h})
    return out


def units(history):
    """User turns as memory units, in chronological order."""
    return [(t["unit_id"], t["content"]) for s in history["sessions"] for t in s if t["role"] == "user"]


In [ ]:
%%writefile /kaggle/working/clm/src/clm/judge.py
"""Stage 4 (GPU): LLM judge over every generation [design_spec 6.2]. Cached in judgments.jsonl."""
import json
import re

from . import config, haystack, prompts

LABEL = re.compile(r"\b(CORRECT|INCORRECT|NO_ANSWER|ANSWERED)\b")


def parse(text):
    m = LABEL.search(text.upper())
    return m.group(1) if m else "UNPARSED"


def run(items_path, backend=None):
    from .llm import Generator
    items = {i["item_id"]: i for i in haystack.load_jsonl(items_path)}
    gens = []
    for p in sorted(config.RESULTS.glob("generations_*.jsonl")):
        gens += haystack.load_jsonl(p)
    out_path = config.RESULTS / "judgments.jsonl"
    done = {(r["id"], r["model"]) for r in haystack.load_jsonl(out_path)} if out_path.exists() else set()
    todo = [g for g in gens if (g["id"], g["model"]) not in done]
    print(f"judge: {len(gens)} generations, {len(todo)} to judge")
    if not todo:
        return
    msgs = []
    for g in todo:
        iid, s, q, qv, setup = g["id"].split("|")
        it = items[iid]
        query = it["text"][q][qv]
        if it.get("is_answerable", True):
            aliases = [a for sl in it["gold"]["slots"] for a in sl["aliases"]]
            msgs.append(prompts.judge(query, it["gold"]["answer_en"], aliases, g["response"]))
        else:
            msgs.append(prompts.judge_abstention(query, g["response"]))
    gen = Generator("judge", backend=backend)
    outs = gen.chat(msgs, max_tokens=8)
    with open(out_path, "a", encoding="utf-8") as f:
        for g, o in zip(todo, outs):
            f.write(json.dumps({"id": g["id"], "model": g["model"], "judge_raw": o, "judge": parse(o)},
                               ensure_ascii=False) + "\n")


In [ ]:
%%writefile /kaggle/working/clm/src/clm/llm.py
"""Chat-generation wrapper: vLLM (default), Hugging Face transformers (fallback), or a mock (local tests).

All backends use greedy decoding (temperature 0) and fp16 [design_spec 5.1]. Qwen3 thinking mode is
disabled through the chat template [5.1].
"""
import os
import re

from . import config

THINK = re.compile(r"<think>.*?</think>", re.S)


def _clean(text):
    return THINK.sub("", text).strip()


class Generator:
    def __init__(self, model_key, backend=None, max_model_len=config.MAX_MODEL_LEN):
        self.model_key = model_key
        self.model_id = config.MODELS[model_key]
        self.backend = backend or os.environ.get("CLM_BACKEND", "vllm")
        self.max_model_len = max_model_len
        if self.backend == "vllm":
            self._init_vllm()
        elif self.backend == "hf":
            self._init_hf()
        elif self.backend != "mock":
            raise ValueError(f"unknown backend {self.backend}")

    # ---------------- vLLM ----------------
    def _init_vllm(self):
        import torch
        from vllm import LLM
        self.llm = LLM(model=self.model_id, dtype="half", tensor_parallel_size=max(1, torch.cuda.device_count()),
                       max_model_len=self.max_model_len, gpu_memory_utilization=0.88,
                       enable_prefix_caching=True, seed=0)

    def _gen_vllm(self, conversations, max_tokens):
        from vllm import SamplingParams
        sp = SamplingParams(temperature=0.0, max_tokens=max_tokens)
        outs = self.llm.chat(conversations, sp, use_tqdm=True,
                             chat_template_kwargs={"enable_thinking": False})
        return [_clean(o.outputs[0].text) for o in outs]

    # ---------------- transformers ----------------
    def _init_hf(self):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        self.tok = AutoTokenizer.from_pretrained(self.model_id, padding_side="left")
        if self.tok.pad_token is None:
            self.tok.pad_token = self.tok.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(self.model_id, torch_dtype=torch.float16, device_map="auto")
        self.model.eval()

    def _gen_hf(self, conversations, max_tokens, batch_size=4):
        import torch
        out = []
        for i in range(0, len(conversations), batch_size):
            batch = conversations[i:i + batch_size]
            texts = [self.tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True,
                                                  enable_thinking=False) for c in batch]
            enc = self.tok(texts, return_tensors="pt", padding=True).to(self.model.device)
            with torch.no_grad():
                gen = self.model.generate(**enc, max_new_tokens=max_tokens, do_sample=False,
                                          pad_token_id=self.tok.pad_token_id)
            for row in gen[:, enc["input_ids"].shape[1]:]:
                out.append(_clean(self.tok.decode(row, skip_special_tokens=True)))
        return out

    # ---------------- mock (CPU tests) ----------------
    def _gen_mock(self, conversations, max_tokens):
        """Echoes the first memory line for readers, copies the text for translators, says CORRECT for judges."""
        out = []
        for c in conversations:
            msg = c[-1]["content"]
            if "Reply with exactly one word" in msg:
                out.append("NO_ANSWER" if "ANSWERED -" in msg else "CORRECT")
            elif msg.startswith(("Translate", "Rewrite", "Write the following")):
                out.append(msg.split("Text:", 1)[-1].strip())
            else:
                m = re.search(r"\n1\. (.+)", msg) or re.search(r"User: (.+)", msg)
                out.append(m.group(1)[:200] if m else "I don't know.")
        return out

    def chat(self, conversations, max_tokens):
        if not conversations:
            return []
        return getattr(self, f"_gen_{self.backend}")(conversations, max_tokens)


In [ ]:
%%writefile /kaggle/working/clm/src/clm/prompts.py
"""Prompt templates, verbatim from design_spec.md Appendix A (plus the filler-translation prompts [3.5])."""

READER_SYSTEM = (
    "You are a personal assistant with memory of your past conversations with this user.\n"
    "Answer the user's question using only what the user told you in past conversations.\n"
    "Answer in one short sentence, in the same language as the question.\n"
    "If the information is not in the past conversations, say that you don't know."
)


def reader_fc(sessions, query):
    blocks = []
    for k, sess in enumerate(sessions, 1):
        lines = [f"[Session {k}]"] + [f"{'User' if t['role'] == 'user' else 'Assistant'}: {t['content']}" for t in sess]
        blocks.append("\n".join(lines))
    user = "Past conversations:\n" + "\n\n".join(blocks) + f"\n\nCurrent question: {query}"
    return [{"role": "system", "content": READER_SYSTEM}, {"role": "user", "content": user}]


def reader_memories(memories, query):
    lines = "\n".join(f"{i}. {m}" for i, m in enumerate(memories, 1))
    user = ("Relevant memories from past conversations (may be incomplete or irrelevant):\n"
            f"{lines}\n\nCurrent question: {query}")
    return [{"role": "system", "content": READER_SYSTEM}, {"role": "user", "content": user}]


def judge(query, gold_en, aliases, response):
    user = (
        "You are grading an assistant's answer about a user's personal information.\n"
        f"Question: {query}\n"
        f"Gold answer: {gold_en}   Accepted forms: {'; '.join(aliases) if aliases else '(none)'}\n"
        f"Assistant answer: {response}\n"
        "The answer may be in English, Hindi (Devanagari) or Hinglish (romanized). Judge meaning, not language.\n"
        "Reply with exactly one word:\n"
        "CORRECT - states the gold information without contradiction\n"
        "INCORRECT - wrong, incomplete, lists several candidates, or contradicts the gold\n"
        "NO_ANSWER - says it does not know / has no information"
    )
    return [{"role": "user", "content": user}]


def judge_abstention(query, response):
    """Unanswerable controls: the only correct behaviour is to say it does not know."""
    user = (
        "You are grading an assistant's answer about a user's personal information.\n"
        f"Question: {query}\n"
        "Gold answer: the user never told the assistant this, so the assistant should say it does not know.\n"
        f"Assistant answer: {response}\n"
        "The answer may be in English, Hindi (Devanagari) or Hinglish (romanized). Judge meaning, not language.\n"
        "Reply with exactly one word:\n"
        "ANSWERED - gives some specific answer to the question\n"
        "NO_ANSWER - says it does not know / has no information"
    )
    return [{"role": "user", "content": user}]


def to_english(text):
    return [{"role": "user", "content":
             "Translate the following text to English. It may be Hindi, Hinglish (romanized Hindi) or English.\n"
             "Keep names, numbers and dates exactly. Output only the translation.\n"
             f"Text: {text}"}]


def to_hindi(text):
    return [{"role": "user", "content":
             "Translate the following text into natural, conversational Hindi written in Devanagari script.\n"
             "Keep brand and product names in Latin script, write person and place names in Devanagari, "
             "and keep digits as 0-9. Output only the translation.\n"
             f"Text: {text}"}]


HINGLISH_INSTRUCTION = (
    "Rewrite the following text in Hinglish: the casual Hindi-English mix used in Indian chat, "
    "with Hindi grammar, common English words mixed in, written ONLY in Roman (Latin) script. "
    "Never use Devanagari. Keep names, numbers and brand names unchanged. Output only the rewritten text.\n"
)

# Few-shot pairs (EN -> HG) from batch01 items F001 (team-verified), F002, P009. Showing the target
# style fixes the first pilot's failure, where Qwen3-8B answered in Devanagari for 77/160 filler turns.
HINGLISH_SHOTS = [
    ("I've been listening to audiobooks during my daily commute, which takes 45 minutes each way. "
     "Can you recommend some good fiction audiobooks?",
     "Main aajkal daily office aate-jaate audiobooks sunti hoon, ek side ka travel 45 minute ka hota hai. "
     "Kya aap kuch acchi fiction audiobooks suggest kar sakte hain?"),
    ("Congratulations on the new role! A simple system helps: scan receipts the same day, keep one folder "
     "per month, and log every expense in a spreadsheet with date, amount and purpose.",
     "Nayi job ki congratulations! Ek simple system follow kijiye: receipts usi din scan karein, har mahine ka "
     "alag folder rakhein, aur har expense ko date, amount aur purpose ke saath ek spreadsheet mein likhein."),
    ("You could use a little less honey and add cinnamon or cardamom for sweetness without sugar.",
     "Honey thoda kam karke sweetness ke liye cinnamon ya elaichi daal sakti hain."),
]


def to_hinglish(text):
    msgs = []
    for en, hg in HINGLISH_SHOTS:
        msgs += [{"role": "user", "content": HINGLISH_INSTRUCTION + f"Text: {en}"},
                 {"role": "assistant", "content": hg}]
    return msgs + [{"role": "user", "content": HINGLISH_INSTRUCTION + f"Text: {text}"}]


def romanize_hindi(hindi_text):
    """Fallback: turn an existing Devanagari Hindi translation into Roman-script Hinglish."""
    return [{"role": "user", "content":
             "Write the following Hindi text in Roman (Latin) script the way Indians type Hindi in chat "
             "(e.g. 'Main kal office gaya tha'). Write common English loanwords in English spelling. "
             "Do not output any Devanagari characters. Output only the romanized text.\n"
             f"Text: {hindi_text}"}]


In [ ]:
%%writefile /kaggle/working/clm/src/clm/report.py
"""Stage 5 (CPU): score every generation and write the analysis tables [design_spec 1, 6].

Outputs in results/<run>/:
  scored.csv                    one row per (model, item, store, query, qv, setup)
  table_accuracy_3x3.csv        accuracy per model x setup x store x query
  table_gap.csv                 H1: monolingual vs cross-lingual gap (paired by item, bootstrap CI)
  table_mirror.csv              H2: mirrored directions, exact McNemar
  table_recall.csv              recall@k per retriever x strategy x store x query
  table_stages.csv              where failures happen: retrieval miss / reading error / oracle error
  table_noise_floor.csv         paraphrase disagreement on monolingual cells
  table_abstention.csv          hallucination rate on unanswerable controls
  table_response_language.csv   answer language by query language
  judge_validation_sample.csv   200 stratified responses for human labelling
  summary.md                    human-readable digest
"""
import csv
import json
import random
from collections import defaultdict

from . import config, haystack
from .textutil import any_match, is_abstention, response_language, slot_match

CROSS = [(s, q) for s in config.LANGS for q in config.LANGS if s != q]
DIAG = [(l, l) for l in config.LANGS]


def mcnemar_exact(b, c):
    """Two-sided exact McNemar p-value from discordant counts b, c."""
    from math import comb
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    p = sum(comb(n, i) for i in range(k + 1)) / 2 ** n
    return min(1.0, 2 * p)


def score_rows(items):
    judg = {(j["id"], j["model"]): j["judge"] for j in haystack.load_jsonl(config.RESULTS / "judgments.jsonl")} \
        if (config.RESULTS / "judgments.jsonl").exists() else {}
    ret = {(r["item_id"], r["store"], r["query"], r["qv"], r["strategy"]): r
           for r in haystack.load_jsonl(config.RESULTS / "retrieval.jsonl") if r["retriever"] == "bge-m3"}
    rows = []
    for p in sorted(config.RESULTS.glob("generations_*.jsonl")):
        for g in haystack.load_jsonl(p):
            iid, s, q, qv, setup = g["id"].split("|")
            it = items[iid]
            ans = it.get("is_answerable", True)
            resp = g["response"]
            j = judg.get((g["id"], g["model"]))
            if ans:
                slots = it["gold"]["slots"]
                correct = (j == "CORRECT") if j else slot_match(resp, slots)
                sm = slot_match(resp, slots)
            else:
                correct = (j == "NO_ANSWER") if j else is_abstention(resp)
                sm = None
            gold_rank = ret[(iid, s, q, qv, setup.split("-")[1])]["gold_rank"] if setup.startswith("RM") and ans else None
            rows.append({
                "model": g["model"], "item_id": iid, "memory_type": it["memory_type"], "answerable": ans,
                "store": s, "query": q, "qv": qv, "setup": setup, "judge": j or "", "correct": int(bool(correct)),
                "slot_match": "" if sm is None else int(sm),
                "distractor_capture": int(any_match(resp, it.get("distractor", {}).get("value_aliases"))) if ans else "",
                "near_miss_capture": int(any_match(resp, it.get("near_miss_aliases"))) if ans else "",
                "abstained": int(is_abstention(resp)), "resp_lang": response_language(resp),
                "gold_rank": gold_rank if gold_rank is not None else "", "response": resp})
    return rows


def write_csv(name, rows):
    if not rows:
        return
    with open(config.RESULTS / name, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)


def mean(xs):
    xs = list(xs)
    return sum(xs) / len(xs) if xs else float("nan")


def item_level(rows, model, setup, cells):
    """Per item: mean correctness over the given cells (both query variants)."""
    acc = defaultdict(list)
    for r in rows:
        if r["model"] == model and r["setup"] == setup and r["answerable"] and (r["store"], r["query"]) in cells:
            acc[r["item_id"]].append(r["correct"])
    return {k: mean(v) for k, v in acc.items()}


def bootstrap_ci(diffs, n=10000, seed=0):
    rng = random.Random(seed)
    if not diffs:
        return float("nan"), float("nan")
    means = sorted(mean(rng.choices(diffs, k=len(diffs))) for _ in range(n))
    return means[int(0.025 * n)], means[int(0.975 * n)]


def run(items_path):
    items = {i["item_id"]: i for i in haystack.load_jsonl(items_path)}
    rows = score_rows(items)
    write_csv("scored.csv", rows)
    models = sorted({r["model"] for r in rows})
    setups = [s for s in config.SETUPS if any(r["setup"] == s for r in rows)]
    ans = [r for r in rows if r["answerable"]]
    md = [f"# Results summary ({config.RESULTS.name})", "",
          f"Generations scored: {len(rows)} | answerable: {len(ans)} | models: {', '.join(models)}", ""]

    # --- 3x3 accuracy ---
    t3 = []
    for m in models:
        for st in setups:
            for s in config.LANGS:
                for q in config.LANGS:
                    xs = [r["correct"] for r in ans if (r["model"], r["setup"], r["store"], r["query"]) == (m, st, s, q)]
                    t3.append({"model": m, "setup": st, "store": s, "query": q, "n": len(xs), "accuracy": round(mean(xs), 4)})
    write_csv("table_accuracy_3x3.csv", t3)
    md += ["## Accuracy (rows = storage language, columns = query language)", ""]
    for m in models:
        for st in setups:
            md += [f"**{m} / {st}**", "", "| store \\ query | EN | HI | HG |", "|---|---|---|---|"]
            for s in config.LANGS:
                cells = [next(x for x in t3 if (x["model"], x["setup"], x["store"], x["query"]) == (m, st, s, q))["accuracy"]
                         for q in config.LANGS]
                md.append(f"| {s.upper()} | " + " | ".join(f"{c:.0%}" for c in cells) + " |")
            md.append("")

    # --- H1 gap (paired by item) ---
    gap = []
    for m in models:
        for st in setups:
            d, c = item_level(ans, m, st, DIAG), item_level(ans, m, st, CROSS)
            diffs = [d[k] - c[k] for k in d if k in c]
            lo, hi = bootstrap_ci(diffs)
            gap.append({"model": m, "setup": st, "n_items": len(diffs), "mono_acc": round(mean(d.values()), 4),
                        "cross_acc": round(mean(c.values()), 4), "gap": round(mean(diffs), 4),
                        "ci95_low": round(lo, 4), "ci95_high": round(hi, 4)})
    write_csv("table_gap.csv", gap)
    md += ["## H1: monolingual minus cross-lingual accuracy (item-paired, 95% bootstrap CI)", "",
           "| model | setup | mono | cross | gap | 95% CI |", "|---|---|---|---|---|---|"]
    md += [f"| {g['model']} | {g['setup']} | {g['mono_acc']:.0%} | {g['cross_acc']:.0%} | {g['gap']*100:+.1f}pp | "
           f"[{g['ci95_low']*100:+.1f}, {g['ci95_high']*100:+.1f}] |" for g in gap] + [""]

    # --- H2 mirrored directions ---
    mir = []
    key = {(r["model"], r["setup"], r["item_id"], r["store"], r["query"], r["qv"]): r["correct"] for r in ans}
    for m in models:
        for st in setups:
            for a, b in (("hi", "en"), ("hg", "en"), ("hg", "hi")):
                n10 = n01 = n = 0
                for iid in items:
                    for qv in config.QUERY_VARIANTS:
                        x, y = key.get((m, st, iid, a, b, qv)), key.get((m, st, iid, b, a, qv))
                        if x is None or y is None:
                            continue
                        n += 1
                        n10 += x and not y
                        n01 += y and not x
                mir.append({"model": m, "setup": st, "direction_A": f"{a}->{b}", "direction_B": f"{b}->{a}", "pairs": n,
                            "A_only_correct": n10, "B_only_correct": n01, "mcnemar_p": round(mcnemar_exact(n10, n01), 4)})
    write_csv("table_mirror.csv", mir)

    # --- Recall@k ---
    rec = []
    rr = [r for r in haystack.load_jsonl(config.RESULTS / "retrieval.jsonl") if r["gold_rank"]]
    for name in sorted({r["retriever"] for r in rr}):
        for strat in ("S1", "S2", "S3"):
            for s in config.LANGS:
                for q in config.LANGS:
                    ranks = [r["gold_rank"] for r in rr if (r["retriever"], r["strategy"], r["store"], r["query"]) == (name, strat, s, q)]
                    rec.append({"retriever": name, "strategy": strat, "store": s, "query": q, "n": len(ranks),
                                **{f"R@{k}": round(mean(int(x <= k) for x in ranks), 4) for k in config.RECALL_KS}})
    write_csv("table_recall.csv", rec)
    md += ["## Recall@5 (bge-m3), mono vs cross", "", "| strategy | mono R@5 | cross R@5 |", "|---|---|---|"]
    for strat in ("S1", "S2", "S3"):
        sub = [x for x in rec if x["retriever"] == "bge-m3" and x["strategy"] == strat]
        md.append(f"| {strat} | {mean(x['R@5'] for x in sub if x['store'] == x['query']):.0%} | "
                  f"{mean(x['R@5'] for x in sub if x['store'] != x['query']):.0%} |")
    md.append("")

    # --- Stage decomposition (RM vs OR) ---
    stg = []
    for m in models:
        for strat in ("S1", "S2", "S3"):
            for s in config.LANGS:
                for q in config.LANGS:
                    rm = [r for r in ans if (r["model"], r["setup"], r["store"], r["query"]) == (m, f"RM-{strat}", s, q)]
                    orc = [r for r in ans if (r["model"], r["setup"], r["store"], r["query"]) == (m, f"OR-{strat}", s, q)]
                    if not rm:
                        continue
                    hit = [r for r in rm if r["gold_rank"] != "" and int(r["gold_rank"]) <= config.TOP_K]
                    stg.append({"model": m, "strategy": strat, "store": s, "query": q, "n": len(rm),
                                "retrieval_miss": round(1 - len(hit) / len(rm), 4),
                                "reading_error_given_hit": round(1 - mean(r["correct"] for r in hit), 4) if hit else "",
                                "oracle_error": round(1 - mean(r["correct"] for r in orc), 4) if orc else "",
                                "end_to_end_error": round(1 - mean(r["correct"] for r in rm), 4)})
    write_csv("table_stages.csv", stg)

    # --- Noise floor (paraphrase disagreement on monolingual cells) ---
    nf = []
    for m in models:
        for st in setups:
            dis = [int(key[(m, st, iid, l, l, "q1")] != key[(m, st, iid, l, l, "q2")])
                   for iid in items for l in config.LANGS
                   if (m, st, iid, l, l, "q1") in key and (m, st, iid, l, l, "q2") in key]
            nf.append({"model": m, "setup": st, "pairs": len(dis), "paraphrase_disagreement": round(mean(dis), 4)})
    write_csv("table_noise_floor.csv", nf)

    # --- Abstention controls ---
    ab = [r for r in rows if not r["answerable"]]
    abt = []
    for m in models:
        for st in setups:
            for s, q in DIAG + CROSS:
                xs = [1 - r["correct"] for r in ab if (r["model"], r["setup"], r["store"], r["query"]) == (m, st, s, q)]
                if xs:
                    abt.append({"model": m, "setup": st, "store": s, "query": q, "n": len(xs), "hallucination_rate": round(mean(xs), 4)})
    write_csv("table_abstention.csv", abt)

    # --- Response language ---
    rl = []
    for m in models:
        for q in config.LANGS:
            sub = [r for r in rows if r["model"] == m and r["query"] == q]
            rl.append({"model": m, "query_lang": q, "n": len(sub),
                       **{f"answered_{l}": round(mean(int(r["resp_lang"] == l) for r in sub), 4) for l in ("en", "hi", "hg", "other")}})
    write_csv("table_response_language.csv", rl)

    # --- Judge sanity + human-validation sample ---
    judged = [r for r in ans if r["judge"]]
    if judged:
        agree = mean(int((r["judge"] == "CORRECT") == bool(r["slot_match"])) for r in judged)
        md += [f"Judge vs slot-match agreement (answerable): {agree:.1%} over {len(judged)} responses "
               "(disagreements are what the human validation must resolve).", ""]
    rng = random.Random(7)
    strata = defaultdict(list)
    for r in rows:
        strata[(r["model"], r["store"], r["query"])].append(r)
    sample = []
    for k in sorted(strata):
        sample += rng.sample(strata[k], min(len(strata[k]), -(-200 // len(strata))))
    write_csv("judge_validation_sample.csv",
              [{"model": r["model"], "item_id": r["item_id"], "store": r["store"], "query": r["query"], "qv": r["qv"],
                "setup": r["setup"], "question": items[r["item_id"]]["text"][r["query"]][r["qv"]],
                "gold": items[r["item_id"]].get("gold", {}).get("answer_en", ""), "response": r["response"],
                "judge": r["judge"], "human_label (CORRECT/INCORRECT/NO_ANSWER)": ""} for r in sample[:200]])
    (config.RESULTS / "summary.md").write_text("\n".join(md), encoding="utf-8")
    print("\n".join(md[:60]))


In [ ]:
%%writefile /kaggle/working/clm/src/clm/retrieve.py
"""Stage 2 (GPU or CPU): rank memory units for every query [design_spec 4.1-4.2].

Strategies (what is indexed / what the query is):
  S1 original : original units            / original query
  S2 pivot    : English translations       / English translation of the query
  S3 dual     : original + English (merged by unit, max score) / original query
Retrievers: bge-m3 (feeds the readers), multilingual-e5 and BM25 (recall only).
"""
import json
import math
import re
from collections import Counter

import numpy as np

from . import config, haystack

TOKEN = re.compile(r"[^\s.,!?;:'\"()\[\]\-–—/।]+")


class DenseEmbedder:
    def __init__(self, model_id, backend="st"):
        self.model_id, self.backend = model_id, backend
        self.e5 = "e5" in model_id
        if backend == "st":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_id, device="cuda" if _has_cuda() else "cpu")
        self.cache = {}

    def _mock(self, text):
        v = np.zeros(512)
        t = text.lower()
        for i in range(len(t) - 2):
            v[hash(t[i:i + 3]) % 512] += 1
        return v

    def embed(self, texts, is_query):
        prefix = ("query: " if is_query else "passage: ") if self.e5 else ""
        need = [t for t in dict.fromkeys(texts) if (prefix + t) not in self.cache]
        if need:
            if self.backend == "st":
                vecs = self.model.encode([prefix + t for t in need], batch_size=32, normalize_embeddings=True,
                                         show_progress_bar=False)
            else:
                vecs = [self._mock(t) for t in need]
                vecs = [v / (np.linalg.norm(v) or 1) for v in vecs]
            for t, v in zip(need, vecs):
                self.cache[prefix + t] = np.asarray(v, dtype=np.float32)
        return np.stack([self.cache[prefix + t] for t in texts])

    def scores(self, query, docs):
        return self.embed(docs, False) @ self.embed([query], True)[0]


class BM25:
    k1, b = 1.5, 0.75

    def scores(self, query, docs):
        toks = [TOKEN.findall(d.lower()) for d in docs]
        avg = sum(map(len, toks)) / max(1, len(toks))
        df = Counter(w for t in toks for w in set(t))
        n = len(docs)
        out = []
        for t in toks:
            tf = Counter(t)
            s = 0.0
            for w in TOKEN.findall(query.lower()):
                if w in tf:
                    idf = math.log(1 + (n - df[w] + 0.5) / (df[w] + 0.5))
                    s += idf * tf[w] * (self.k1 + 1) / (tf[w] + self.k1 * (1 - self.b + self.b * len(t) / avg))
            out.append(s)
        return np.array(out)


def _has_cuda():
    try:
        import torch
        return torch.cuda.is_available()
    except ImportError:
        return False


def rank(retriever, query, units, strategy, en):
    """Return unit ids ranked best-first."""
    ids = [u for u, _ in units]
    if strategy == "S1":
        sc = retriever.scores(query, [x for _, x in units])
    elif strategy == "S2":
        sc = retriever.scores(en.get(query, query), [en.get(x, x) for _, x in units])
    else:  # S3 dual index: score both representations, keep the max per unit
        a = retriever.scores(query, [x for _, x in units])
        b = retriever.scores(query, [en.get(x, x) for _, x in units])
        sc = np.maximum(a, b)
    order = np.argsort(-np.asarray(sc), kind="stable")
    return [ids[i] for i in order]


def run(items_path, backend=None):
    backend = backend or ("mock" if __import__("os").environ.get("CLM_BACKEND") == "mock" else "st")
    items = haystack.load_jsonl(items_path)
    hist = {(h["item_id"], h["lang"]): h for h in haystack.load_jsonl(config.RESULTS / "histories.jsonl")}
    en = json.loads((config.RESULTS / "en_translations.json").read_text(encoding="utf-8"))
    retrievers = {"bge-m3": DenseEmbedder(config.EMBEDDER, backend),
                  "me5": DenseEmbedder(config.EMBEDDER_2, backend),
                  "bm25": BM25()}
    rows = []
    for it in items:
        for s in config.LANGS:
            h = hist[(it["item_id"], s)]
            units = haystack.units(h)
            for q in config.LANGS:
                for qv in config.QUERY_VARIANTS:
                    query = it["text"][q][qv]
                    for strat in ("S1", "S2", "S3"):
                        for name, r in retrievers.items():
                            ranked = rank(r, query, units, strat, en)
                            gold = h["gold_unit"] if it.get("is_answerable", True) else None
                            rows.append({"item_id": it["item_id"], "store": s, "query": q, "qv": qv,
                                         "strategy": strat, "retriever": name, "ranked": ranked[:10],
                                         "gold_rank": ranked.index(gold) + 1 if gold else None,
                                         "distractor_rank": (ranked.index(h["distractor_unit"]) + 1
                                                             if h["distractor_unit"] else None)})
    with open(config.RESULTS / "retrieval.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    print(f"retrieval stage: {len(rows)} rankings")


In [ ]:
%%writefile /kaggle/working/clm/src/clm/stages.py
"""Command-line entry point. Each GPU stage runs in its own process so GPU memory is fully released.

  python -m clm.stages translate --items data/items/items_batch01_pilot.jsonl
  python -m clm.stages retrieve  --items ...
  python -m clm.stages tokens    --items ...
  python -m clm.stages generate  --items ... --model qwen3-8b
  python -m clm.stages generate  --items ... --model llama3.1-8b
  python -m clm.stages judge     --items ...
  python -m clm.stages report    --items ...
Set CLM_BACKEND=mock to test the whole pipeline on a CPU; CLM_RUN names the results folder.
"""
import argparse

from . import config


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("stage", choices=["translate", "retrieve", "tokens", "generate", "judge", "report"])
    ap.add_argument("--items", required=True)
    ap.add_argument("--model", default=None)
    ap.add_argument("--backend", default=None, help="vllm | hf | mock (default: env CLM_BACKEND or vllm)")
    ap.add_argument("--setups", default=",".join(config.SETUPS))
    a = ap.parse_args()
    config.RESULTS.mkdir(parents=True, exist_ok=True)
    if a.stage == "translate":
        from . import translate
        translate.run(a.items, a.backend)
    elif a.stage == "retrieve":
        from . import retrieve
        retrieve.run(a.items)
    elif a.stage == "tokens":
        from . import tokens
        tokens.run(a.items)
    elif a.stage == "generate":
        from . import generate
        generate.run(a.items, a.model, a.backend, tuple(a.setups.split(",")))
    elif a.stage == "judge":
        from . import judge
        judge.run(a.items, a.backend)
    elif a.stage == "report":
        from . import report
        report.run(a.items)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/clm/src/clm/textutil.py
"""Normalization and deterministic checks used in scoring [design_spec 6.1]."""
import re
import unicodedata

DEV_DIGITS = str.maketrans("०१२३४५६७८९", "0123456789")
NUKTA = "़"
DEVANAGARI = re.compile(r"[ऀ-ॿ]")
LATIN = re.compile(r"[A-Za-z]")

# Common Hindi function words in Roman script: their presence marks Hinglish rather than English.
HG_MARKERS = {"hai", "hain", "tha", "thi", "nahi", "nahin", "kya", "mera", "meri", "mere", "mujhe", "aap", "aapka",
              "aapki", "ka", "ki", "ke", "ko", "se", "mein", "aur", "ya", "toh", "hoon", "kar", "karti", "karta",
              "liya", "gaya", "gayi", "pe", "par", "wala", "wali", "kab", "kahan", "kitna", "kitne", "pata"}

ABSTAIN = re.compile(
    r"(don't know|do not know|not sure|no information|not (been )?mentioned|didn't (mention|tell|say)|"
    r"haven't (mentioned|told)|not in (our|the|my) (past )?conversations?|no record|cannot (find|determine)|"
    r"can't (find|determine)|pata nahi|nahi pata|nahin pata|jaankari nahi|maloom nahi|"
    r"नहीं पता|पता नहीं|जानकारी नहीं|उल्लेख नहीं|मालूम नहीं|नहीं बताया)", re.I)


def norm(s):
    s = unicodedata.normalize("NFC", s or "").translate(DEV_DIGITS).replace(NUKTA, "")
    s = s.replace("’", "'").replace("–", "-").replace("—", "-")
    return s.casefold()


def contains(text, alias):
    """Word-bounded match for Latin aliases, substring match for Devanagari (after normalization)."""
    t, a = norm(text), norm(alias)
    if not a:
        return False
    if re.search(r"[a-z0-9]", a):
        return re.search(rf"(?<![a-z0-9]){re.escape(a)}(?![a-z0-9])", t) is not None
    return a in t


def slot_match(response, slots):
    """All gold slots present (any alias, any script)."""
    return bool(slots) and all(any(contains(response, a) for a in s["aliases"]) for s in slots)


def any_match(response, aliases):
    return any(contains(response, a) for a in aliases or [])


def is_abstention(response):
    return bool(ABSTAIN.search(response or ""))


def response_language(text):
    """'hi' (Devanagari), 'hg' (romanized Hindi), 'en', or 'other'."""
    dev, lat = len(DEVANAGARI.findall(text or "")), len(LATIN.findall(text or ""))
    if dev + lat == 0:
        return "other"
    if dev / (dev + lat) > 0.3:
        return "hi"
    words = re.findall(r"[a-z]+", (text or "").lower())
    hg = sum(w in HG_MARKERS for w in words)
    return "hg" if words and (hg >= 2 or hg / len(words) > 0.15) else "en"


def devanagari_share(text):
    dev, lat = len(DEVANAGARI.findall(text or "")), len(LATIN.findall(text or ""))
    return dev / (dev + lat) if dev + lat else 0.0


In [ ]:
%%writefile /kaggle/working/clm/src/clm/tokens.py
"""Token counts of full-context prompts per storage language and tokenizer [design_spec 3.5 CHECK IN PHASE 4].

Hindi/Hinglish need more tokens than English for the same content; we report this and confirm every
prompt fits MAX_MODEL_LEN. Writes table_tokens.csv.
"""
import csv

from . import config, generate, haystack


def run(items_path):
    from transformers import AutoTokenizer
    items = haystack.load_jsonl(items_path)
    reqs = [r for r in generate.build_requests(items, setups=("FC",)) if r["id"].split("|")[3] == "q1"]
    rows = []
    for key in ("qwen3-8b", "llama3.1-8b"):
        tok = AutoTokenizer.from_pretrained(config.MODELS[key])
        by_lang = {}
        for r in reqs:
            # Render to text, then tokenize: in transformers 5.x apply_chat_template(tokenize=True) returns a
            # dict, and len() of it was 2 in the first pilot. The template already holds special tokens.
            text = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=True)
            n = len(tok(text, add_special_tokens=False)["input_ids"])
            by_lang.setdefault(r["id"].split("|")[1], []).append(n)
        for lang, ns in sorted(by_lang.items()):
            rows.append({"tokenizer": key, "store_lang": lang, "prompts": len(ns), "mean_tokens": round(sum(ns) / len(ns)),
                         "max_tokens": max(ns), "fits_context": max(ns) + config.READER_MAX_TOKENS <= config.MAX_MODEL_LEN})
    with open(config.RESULTS / "table_tokens.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    for r in rows:
        print(r)
    if not all(r["fits_context"] for r in rows):
        raise RuntimeError("Some full-context prompts exceed MAX_MODEL_LEN - see table_tokens.csv")


In [ ]:
%%writefile /kaggle/working/clm/src/clm/translate.py
"""Stage 1 (GPU, fixed translator Qwen3-8B) [design_spec 3.5, 4.3].

1. Translate the 40 English filler sessions into HI and HG (all turns).
2. Build every history (item x storage language).
3. Translate every HI/HG memory unit and query into English (for S2/S3), cached in en_translations.json.
"""
import json

from . import config, haystack, prompts
from .textutil import devanagari_share


def _write_jsonl(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def translate_fillers(gen, log):
    pool_en = haystack.load_jsonl(config.DATA / "haystacks" / "filler_pool_en.jsonl")
    texts = [t["content"] for s in pool_en for t in s["turns"]]
    hi = gen.chat([prompts.to_hindi(x) for x in texts], max_tokens=400)
    hg = gen.chat([prompts.to_hinglish(x) for x in texts], max_tokens=400)
    # Hinglish must be Roman script. Fallback for turns where Devanagari leaked in: romanize the
    # Hindi translation of the same turn (an easier task than translating from English).
    bad = [i for i, x in enumerate(hg) if devanagari_share(x) > 0.1]
    if bad:
        retry = gen.chat([prompts.romanize_hindi(hi[i]) for i in bad], max_tokens=400)
        for i, x in zip(bad, retry):
            hg[i] = x
    still = [i for i, x in enumerate(hg) if devanagari_share(x) > 0.1]
    log["filler_turns"] = len(texts)
    log["hinglish_fewshot_failures"] = len(bad)
    log["hinglish_still_devanagari"] = len(still)
    log["hindi_low_devanagari"] = sum(devanagari_share(x) < 0.5 for x in hi)
    print("filler translation:", log)
    # Fail loudly rather than silently building half-Devanagari "Hinglish" histories [design_spec 3.5].
    # (Skipped for the mock backend, whose "translations" are English copies by design.)
    if gen.backend != "mock" and (len(still) > 0.05 * len(texts) or log["hindi_low_devanagari"] > 0.05 * len(texts)):
        raise RuntimeError(f"Filler translation quality check failed: {log}. Hinglish/Hindi fillers not saved.")
    for lang, outs in (("hi", hi), ("hg", hg)):
        it = iter(outs)
        rows = [{**s, "turns": [{"role": t["role"], "content": next(it)} for t in s["turns"]]} for s in pool_en]
        _write_jsonl(haystack.filler_path(lang), rows)


def run(items_path, backend=None):
    from .llm import Generator
    gen = Generator(config.TRANSLATOR, backend=backend)
    log = {}
    if not (haystack.filler_path("hi").exists() and haystack.filler_path("hg").exists()):
        translate_fillers(gen, log)
    items = haystack.load_jsonl(items_path)
    hist = haystack.build_all(items, haystack.load_fillers())
    _write_jsonl(config.RESULTS / "histories.jsonl", hist)

    cache_path = config.RESULTS / "en_translations.json"
    cache = json.loads(cache_path.read_text(encoding="utf-8")) if cache_path.exists() else {}
    todo = set()
    for h in hist:
        if h["lang"] != "en":
            todo.update(text for _, text in haystack.units(h))
    for it in items:
        for lang in ("hi", "hg"):
            todo.update(it["text"][lang][q] for q in config.QUERY_VARIANTS)
    todo = sorted(x for x in todo if x not in cache)
    for src, out in zip(todo, gen.chat([prompts.to_english(x) for x in todo], max_tokens=300)):
        cache[src] = out
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cache_path.write_text(json.dumps(cache, ensure_ascii=False, indent=0), encoding="utf-8")
    log["en_translations"] = len(cache)
    (config.RESULTS / "log_translate.json").write_text(json.dumps(log, indent=2))
    print("translate stage:", log)


In [ ]:
%%writefile /kaggle/working/clm/data/items/items_batch01_pilot.jsonl
{"item_id": "F001", "cand_id": "FC106", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I've been listening to audiobooks during my daily commute, which takes 45 minutes each way. Can you recommend some good fiction audiobooks?", "assistant_reply": "That's a great way to use your travel time! A few popular picks: The Nightingale by Kristin Hannah, The Silent Patient by Alex Michaelides, and Project Hail Mary by Andy Weir. Do you prefer mysteries or historical fiction?", "followup_turn": "I'm already reading Gone Girl on my Kindle, but The Nightingale sounds interesting. What is it about?", "followup_reply": "It's a historical novel set in France during World War II, following two sisters who resist the occupation in very different ways.", "q1": "How long does my commute to work take, one way?", "q2": "One way, how much time does it take me to get to work?", "distractor_sentence": "By the way, my sister's commute is only 20 minutes each way, lucky her!"}, "hi": {"memory_turn": "मैं आजकल रोज़ ऑफिस आते-जाते ऑडियोबुक्स सुनती हूँ, एक तरफ़ का सफ़र 45 मिनट का होता है। क्या आप कुछ अच्छी फिक्शन ऑडियोबुक्स बता सकते हैं?", "assistant_reply": "सफ़र के समय का यह बढ़िया इस्तेमाल है! कुछ लोकप्रिय विकल्प: क्रिस्टिन हैना की The Nightingale, एलेक्स माइकलिडीज़ की The Silent Patient और एंडी वियर की Project Hail Mary। आपको मिस्ट्री पसंद है या ऐतिहासिक कहानियाँ?", "followup_turn": "मैं Gone Girl पहले से अपने Kindle पर पढ़ रही हूँ, पर The Nightingale दिलचस्प लग रही है। यह किस बारे में है?", "followup_reply": "यह दूसरे विश्व युद्ध के दौरान फ़्रांस में बसी एक ऐतिहासिक कहानी है, जिसमें दो बहनें अलग-अलग तरीक़ों से कब्ज़े का विरोध करती हैं।", "q1": "मेरे ऑफिस जाने में एक तरफ़ कितना समय लगता है?", "q2": "एक तरफ़ से, मुझे काम पर पहुँचने में कितनी देर लगती है?", "distractor_sentence": "वैसे, मेरी बहन का ऑफिस तो बस एक तरफ़ 20 मिनट दूर है, किस्मत वाली है!"}, "hg": {"memory_turn": "Main aajkal daily office aate-jaate audiobooks sunti hoon, ek side ka travel 45 minute ka hota hai. Kya aap kuch acchi fiction audiobooks suggest kar sakte hain?", "assistant_reply": "Travel time ka yeh bahut accha use hai! Kuch popular options: Kristin Hannah ki The Nightingale, Alex Michaelides ki The Silent Patient, aur Andy Weir ki Project Hail Mary. Aapko mystery pasand hai ya historical fiction?", "followup_turn": "Main Gone Girl already apne Kindle pe padh rahi hoon, par The Nightingale interesting lag rahi hai. Yeh kis baare mein hai?", "followup_reply": "Yeh World War II ke time France mein set ek historical novel hai, jismein do behnein alag-alag tareeke se occupation ka virodh karti hain.", "q1": "Mera office jaane mein ek side kitna time lagta hai?", "q2": "One way, mujhe kaam pe pahunchne mein kitni der lagti hai?", "distractor_sentence": "Waise, meri behen ka office toh bas ek side 20 minute door hai, lucky hai woh!"}}, "gold": {"answer_en": "45 minutes (each way)", "slots": [{"name": "duration", "aliases": ["45 minutes", "45 minute", "45 mins", "45 min", "45 मिनट", "पौन घंटा", "paun ghanta", "three quarters of an hour"]}]}, "distractor": {"value_aliases": ["20 minutes", "20 minute", "20 min", "20 मिनट"]}, "source": {"dataset": "LongMemEval", "question_id": "118b2229", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 1599741519, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F002", "cand_id": "FC105", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I graduated with a degree in Business Administration, and it has really helped in my new role. Any advice on staying organized with paperwork, especially tracking expenses and reimbursements?", "assistant_reply": "Congratulations on the new role! A simple system helps: scan receipts the same day, keep one folder per month, and log every expense in a spreadsheet with date, amount and purpose.", "followup_turn": "A spreadsheet sounds good. Should I use Excel or Google Sheets for this?", "followup_reply": "Either works. Google Sheets is handy if you want to update it from your phone and share it with your manager; Excel is better for heavier analysis offline.", "q1": "What degree did I graduate with?", "q2": "What subject is my degree in?"}, "hi": {"memory_turn": "मैंने बिज़नेस एडमिनिस्ट्रेशन में डिग्री ली है, और इससे मुझे अपनी नई नौकरी में बहुत मदद मिली है। कागज़ात सँभालकर रखने के लिए कोई सलाह देंगे, ख़ासकर ख़र्चों और रीइम्बर्समेंट का हिसाब रखने के लिए?", "assistant_reply": "नई नौकरी की बधाई! एक आसान तरीक़ा अपनाइए: रसीदें उसी दिन स्कैन करें, हर महीने का एक अलग फ़ोल्डर रखें, और हर ख़र्च को तारीख़, रक़म और मक़सद के साथ एक स्प्रेडशीट में लिखें।", "followup_turn": "स्प्रेडशीट अच्छा आइडिया है। इसके लिए Excel ठीक रहेगा या Google Sheets?", "followup_reply": "दोनों चलेंगे। अगर आप फ़ोन से अपडेट करना और मैनेजर के साथ शेयर करना चाहते हैं तो Google Sheets आसान है; ऑफ़लाइन ज़्यादा विश्लेषण के लिए Excel बेहतर है।", "q1": "मैंने किस विषय में डिग्री ली है?", "q2": "मेरी ग्रेजुएशन की डिग्री किस विषय में है?"}, "hg": {"memory_turn": "Maine Business Administration mein degree li hai, aur isse meri nayi job mein bahut help mili hai. Paperwork organized rakhne ke liye koi advice denge, especially expenses aur reimbursements track karne ke liye?", "assistant_reply": "Nayi job ki congratulations! Ek simple system follow kijiye: receipts usi din scan karein, har mahine ka alag folder rakhein, aur har expense ko date, amount aur purpose ke saath ek spreadsheet mein likhein.", "followup_turn": "Spreadsheet accha idea hai. Iske liye Excel theek rahega ya Google Sheets?", "followup_reply": "Dono chalenge. Agar aap phone se update karna aur manager ke saath share karna chahte hain toh Google Sheets easy hai; offline zyada analysis ke liye Excel better hai.", "q1": "Maine kis subject mein degree li hai?", "q2": "Meri graduation ki degree kis subject mein hai?"}}, "gold": {"answer_en": "Business Administration", "slots": [{"name": "degree", "aliases": ["Business Administration", "BBA", "बिज़नेस एडमिनिस्ट्रेशन", "बिजनेस एडमिनिस्ट्रेशन", "व्यवसाय प्रशासन"]}]}, "source": {"dataset": "LongMemEval", "question_id": "e47becba", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 5, "filler_seed": 42481310, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F003", "cand_id": "FC110", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I need to update my address with my health insurance provider. Also, I recently changed my last name and I'm still getting used to it. My old surname was Johnson, but now it's Winters. Does that need a separate update?", "assistant_reply": "Yes, a name change is usually a separate request. Insurers typically ask for a copy of the legal document, such as a marriage certificate or court order, along with your policy number.", "followup_turn": "Okay. Can I change the address online, or do I have to call them?", "followup_reply": "Most insurers let you change your address online through the member portal or app. Calling is only needed if the portal does not allow edits for your plan.", "q1": "What was my last name before I changed it?", "q2": "What was my old surname?"}, "hi": {"memory_turn": "मुझे अपनी हेल्थ इंश्योरेंस कंपनी में अपना पता अपडेट करवाना है। साथ ही, मैंने हाल ही में अपना सरनेम बदला है और अभी उसकी आदत डाल रही हूँ। मेरा पुराना सरनेम जॉनसन था, अब विंटर्स है। क्या इसके लिए अलग से अपडेट करना होगा?", "assistant_reply": "हाँ, नाम बदलने का अनुरोध आमतौर पर अलग से होता है। इंश्योरेंस कंपनियाँ अक्सर क़ानूनी दस्तावेज़ की कॉपी माँगती हैं, जैसे विवाह प्रमाणपत्र या कोर्ट का आदेश, साथ में आपका पॉलिसी नंबर।", "followup_turn": "ठीक है। क्या पता ऑनलाइन बदल सकती हूँ, या उन्हें फ़ोन करना पड़ेगा?", "followup_reply": "ज़्यादातर कंपनियाँ मेंबर पोर्टल या ऐप से ऑनलाइन पता बदलने देती हैं। फ़ोन तभी करना पड़ता है जब आपके प्लान में पोर्टल पर बदलाव की सुविधा न हो।", "q1": "सरनेम बदलने से पहले मेरा सरनेम क्या था?", "q2": "मेरा पुराना सरनेम क्या था?"}, "hg": {"memory_turn": "Mujhe apni health insurance company mein apna address update karwana hai. Saath hi, maine recently apna surname change kiya hai aur abhi uski aadat daal rahi hoon. Mera purana surname Johnson tha, ab Winters hai. Kya iske liye alag se update karna hoga?", "assistant_reply": "Haan, name change ki request usually alag hoti hai. Insurance companies aksar legal document ki copy maangti hain, jaise marriage certificate ya court order, saath mein aapka policy number.", "followup_turn": "Theek hai. Kya address online change kar sakti hoon, ya unhe call karna padega?", "followup_reply": "Zyadatar companies member portal ya app se online address change karne deti hain. Call tabhi karna padta hai jab aapke plan mein portal pe changes ki facility na ho.", "q1": "Surname change karne se pehle mera surname kya tha?", "q2": "Mera purana surname kya tha?"}}, "gold": {"answer_en": "Johnson", "slots": [{"name": "old_surname", "aliases": ["Johnson", "जॉनसन", "जानसन"]}]}, "near_miss_aliases": ["Winters", "विंटर्स"], "source": {"dataset": "LongMemEval", "question_id": "c5e8278d", "lme_type": "single-session-user"}, "topic": "fitness/health", "is_answerable": true, "evidence_position": 9, "filler_seed": 4209568809, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F004", "cand_id": "FC118", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been to the Great Ocean Road before. I went there with friends during my study abroad program at the University of Melbourne, and we spent the whole day hiking along the coast. Any other must-see places in Australia?", "assistant_reply": "Lucky you, that drive is stunning! If you go back, consider the Blue Mountains near Sydney, the Great Barrier Reef in Queensland, and Uluru in the Northern Territory.", "followup_turn": "Uluru sounds amazing. What's the best season to visit it?", "followup_reply": "Roughly May to September is best. Days are mild then, while summer temperatures there often go above 40 degrees Celsius.", "q1": "At which university did I do my study abroad program?", "q2": "Where did I study during my study abroad program?"}, "hi": {"memory_turn": "मैं पहले ग्रेट ओशन रोड जा चुका हूँ। अपने स्टडी अब्रॉड प्रोग्राम के दौरान, जो यूनिवर्सिटी ऑफ़ मेलबर्न में था, मैं दोस्तों के साथ वहाँ गया था और हमने पूरे दिन तट के किनारे हाइकिंग की। ऑस्ट्रेलिया में और कौन-सी जगहें ज़रूर देखनी चाहिए?", "assistant_reply": "आप ख़ुशक़िस्मत हैं, वह रास्ता बेहद ख़ूबसूरत है! अगर दोबारा जाएँ तो सिडनी के पास ब्लू माउंटेंस, क्वींसलैंड में ग्रेट बैरियर रीफ़ और नॉर्दर्न टेरिटरी में उलुरु देखिए।", "followup_turn": "उलुरु कमाल का लगता है। वहाँ जाने का सबसे अच्छा मौसम कौन-सा है?", "followup_reply": "मई से सितंबर के बीच का समय सबसे अच्छा है। तब दिन सुहावने रहते हैं, जबकि गर्मियों में वहाँ तापमान अक्सर 40 डिग्री सेल्सियस से ऊपर चला जाता है।", "q1": "मैंने अपना स्टडी अब्रॉड प्रोग्राम किस यूनिवर्सिटी में किया था?", "q2": "स्टडी अब्रॉड के दौरान मैं कहाँ पढ़ा था?"}, "hg": {"memory_turn": "Main pehle Great Ocean Road ja chuka hoon. Apne study abroad program ke time, jo University of Melbourne mein tha, main dosto ke saath wahan gaya tha aur humne poore din coast ke kinare hiking ki. Australia mein aur kaunsi jagah zaroor dekhni chahiye?", "assistant_reply": "Aap lucky hain, woh drive bahut khoobsurat hai! Agar dobara jaayein toh Sydney ke paas Blue Mountains, Queensland mein Great Barrier Reef aur Northern Territory mein Uluru dekhiye.", "followup_turn": "Uluru kamaal ka lagta hai. Wahan jaane ka best season kaunsa hai?", "followup_reply": "May se September ke beech ka time sabse accha hai. Tab din pleasant rehte hain, jabki garmiyon mein wahan temperature aksar 40 degree Celsius se upar chala jaata hai.", "q1": "Maine apna study abroad program kis university mein kiya tha?", "q2": "Study abroad ke time main kahan padha tha?"}}, "gold": {"answer_en": "University of Melbourne", "slots": [{"name": "university", "aliases": ["University of Melbourne", "Melbourne University", "Melbourne", "यूनिवर्सिटी ऑफ़ मेलबर्न", "यूनिवर्सिटी ऑफ मेलबर्न", "मेलबर्न यूनिवर्सिटी", "मेलबर्न"]}]}, "source": {"dataset": "LongMemEval", "question_id": "3b6f954b", "lme_type": "single-session-user"}, "topic": "work/study", "is_answerable": true, "evidence_position": 2, "filler_seed": 2679826000, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F005", "cand_id": "FC121", "memory_type": "fact", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "For my sister's birthday, I got her a yellow dress and matching earrings. Now I want to plan a small dinner for her at home. Any ideas for a cozy menu?", "assistant_reply": "How thoughtful! Try a simple starter like tomato soup, a main such as pasta or a paneer dish, and finish with a homemade dessert like chocolate mousse.", "followup_turn": "She loves Italian food, so pasta sounds perfect. Which sauce is easy but impressive?", "followup_reply": "A creamy garlic and mushroom sauce is quick and feels special. Finish it with fresh parsley and grated parmesan just before serving.", "q1": "What did I buy for my sister's birthday?", "q2": "What gift did I get my sister for her birthday?"}, "hi": {"memory_turn": "अपनी बहन के जन्मदिन पर मैंने उसके लिए एक पीली ड्रेस और उसके साथ मैचिंग इयररिंग्स लिए। अब मैं उसके लिए घर पर एक छोटा-सा डिनर रखना चाहती हूँ। कोई अच्छा मेन्यू सुझाएँगे?", "assistant_reply": "कितना प्यारा ख़याल है! शुरुआत में टमाटर सूप, मेन कोर्स में पास्ता या पनीर की कोई डिश, और आख़िर में घर पर बना चॉकलेट मूस रख सकती हैं।", "followup_turn": "उसे इटैलियन खाना बहुत पसंद है, तो पास्ता बढ़िया रहेगा। कौन-सी सॉस आसान भी हो और ख़ास भी लगे?", "followup_reply": "क्रीमी गार्लिक मशरूम सॉस जल्दी बनती है और ख़ास भी लगती है। परोसने से ठीक पहले ऊपर से ताज़ा पार्सले और कद्दूकस किया पार्मेज़ान डालें।", "q1": "मैंने अपनी बहन के जन्मदिन पर उसके लिए क्या ख़रीदा था?", "q2": "बहन के बर्थडे पर मैंने उसे क्या गिफ़्ट दिया था?"}, "hg": {"memory_turn": "Apni behen ke birthday pe maine uske liye ek yellow dress aur uske saath matching earrings li. Ab main uske liye ghar pe ek chhota sa dinner rakhna chahti hoon. Koi accha menu suggest karenge?", "assistant_reply": "Kitna pyaara idea hai! Starter mein tomato soup, main course mein pasta ya koi paneer dish, aur end mein ghar ka bana chocolate mousse rakh sakti hain.", "followup_turn": "Use Italian food bahut pasand hai, toh pasta perfect rahega. Kaunsi sauce easy bhi ho aur special bhi lage?", "followup_reply": "Creamy garlic mushroom sauce jaldi banti hai aur special bhi lagti hai. Serve karne se theek pehle upar se fresh parsley aur grated parmesan daalein.", "q1": "Maine apni behen ke birthday pe uske liye kya khareeda tha?", "q2": "Behen ke birthday pe maine use kya gift diya tha?"}}, "gold": {"answer_en": "a yellow dress (and matching earrings)", "slots": [{"name": "gift", "aliases": ["yellow dress", "पीली ड्रेस", "पीली पोशाक", "peeli dress", "pili dress"]}]}, "source": {"dataset": "LongMemEval", "question_id": "66f24dbb", "lme_type": "single-session-user"}, "topic": "family/social", "is_answerable": true, "evidence_position": 5, "filler_seed": 547367346, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F006", "cand_id": "FC128", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm thinking of getting Max a new collar with a name tag. Do you have any recommendations for a good collar for a Golden Retriever like him?", "assistant_reply": "For a medium-to-large dog, a padded nylon or leather collar with a sturdy metal buckle works well. Make sure you can fit two fingers under it once it is on.", "followup_turn": "Should the name tag have my phone number on it?", "followup_reply": "Yes, your phone number is the most useful detail. Many owners also add a short line saying the dog is microchipped, so whoever finds him knows to get it scanned.", "q1": "What breed is my dog?", "q2": "Which breed is Max?"}, "hi": {"memory_turn": "मैं मैक्स के लिए नेम टैग वाला एक नया कॉलर लेने की सोच रहा हूँ। उसके जैसे गोल्डन रिट्रीवर के लिए कोई अच्छा कॉलर सुझाएँगे?", "assistant_reply": "मध्यम से बड़े कुत्ते के लिए गद्देदार नायलॉन या चमड़े का कॉलर, मज़बूत धातु के बकल के साथ, अच्छा रहता है। पहनाने के बाद ध्यान रखें कि उसके नीचे दो उँगलियाँ आ सकें।", "followup_turn": "क्या नेम टैग पर मेरा फ़ोन नंबर होना चाहिए?", "followup_reply": "हाँ, फ़ोन नंबर सबसे काम की जानकारी है। कई लोग एक छोटी-सी लाइन भी लिखते हैं कि कुत्ते में माइक्रोचिप है, ताकि जिसे वह मिले वह उसे स्कैन करवा सके।", "q1": "मेरा कुत्ता किस नस्ल का है?", "q2": "मैक्स किस नस्ल का है?"}, "hg": {"memory_turn": "Main Max ke liye name tag wala ek naya collar lene ki soch raha hoon. Uske jaise Golden Retriever ke liye koi accha collar suggest karenge?", "assistant_reply": "Medium se bade dog ke liye padded nylon ya leather collar, strong metal buckle ke saath, accha rehta hai. Pehnane ke baad dhyan rakhein ki uske neeche do ungliyan aa sakein.", "followup_turn": "Kya name tag pe mera phone number hona chahiye?", "followup_reply": "Haan, phone number sabse kaam ki detail hai. Kai log ek chhoti si line bhi likhte hain ki dog microchipped hai, taaki jise woh mile woh use scan karwa sake.", "q1": "Mera dog kis breed ka hai?", "q2": "Max kis breed ka hai?"}}, "gold": {"answer_en": "Golden Retriever", "slots": [{"name": "breed", "aliases": ["Golden Retriever", "गोल्डन रिट्रीवर", "golden retriver"]}]}, "source": {"dataset": "LongMemEval", "question_id": "75499fd8", "lme_type": "single-session-user"}, "topic": "pets", "is_answerable": true, "evidence_position": 9, "filler_seed": 2344873239, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F007", "cand_id": "FC131", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "We've made a lot of changes at home recently, and my cat Luna has been such a sweetie through all of it. How can I tell if a cat is stressed by changes in her environment?", "assistant_reply": "Common signs include hiding more than usual, changes in appetite, over-grooming, and going outside the litter box. Keeping her feeding and play routine steady helps a lot.", "followup_turn": "She has been hiding under the bed a bit more. Should I try to pull her out?", "followup_reply": "No, let her come out on her own. Leave a soft blanket and some treats nearby, and spend quiet time in the room so she learns the new setup is safe.", "q1": "What is my cat's name?", "q2": "What's my cat called?", "distractor_sentence": "By the way, my neighbour's cat Coco keeps visiting our balcony."}, "hi": {"memory_turn": "हाल ही में हमने घर में काफ़ी बदलाव किए हैं, और मेरी बिल्ली लूना इस पूरे दौरान बहुत प्यारी बनी रही। कैसे पता करूँ कि माहौल बदलने से बिल्ली तनाव में है?", "assistant_reply": "आम संकेत हैं: सामान्य से ज़्यादा छिपना, भूख में बदलाव, बार-बार ख़ुद को चाटना, और लिटर बॉक्स के बाहर गंदगी करना। खाने और खेलने का रूटीन एक जैसा रखने से काफ़ी मदद मिलती है।", "followup_turn": "वह आजकल बेड के नीचे थोड़ा ज़्यादा छिप रही है। क्या उसे बाहर निकालूँ?", "followup_reply": "नहीं, उसे ख़ुद बाहर आने दीजिए। पास में एक नरम कंबल और कुछ ट्रीट्स रखें, और उस कमरे में शांति से समय बिताएँ ताकि उसे लगे कि नया माहौल सुरक्षित है।", "q1": "मेरी बिल्ली का नाम क्या है?", "q2": "मैंने अपनी बिल्ली का क्या नाम रखा है?", "distractor_sentence": "वैसे, पड़ोसियों की बिल्ली कोको बार-बार हमारी बालकनी में आ जाती है।"}, "hg": {"memory_turn": "Recently humne ghar mein kaafi changes kiye hain, aur meri cat Luna is poore time bahut sweet rahi. Kaise pata karoon ki environment badalne se cat stressed hai?", "assistant_reply": "Common signs hain: normal se zyada chhupna, appetite mein change, baar-baar khud ko chaatna, aur litter box ke bahar gandagi karna. Khaane aur khelne ka routine same rakhne se kaafi help milti hai.", "followup_turn": "Woh aajkal bed ke neeche thoda zyada chhup rahi hai. Kya use bahar nikaalun?", "followup_reply": "Nahi, use khud bahar aane dijiye. Paas mein ek soft blanket aur kuch treats rakhein, aur us room mein shaanti se time spend karein taaki use lage ki naya setup safe hai.", "q1": "Meri cat ka naam kya hai?", "q2": "Maine apni cat ka kya naam rakha hai?", "distractor_sentence": "Waise, padosiyon ki cat Coco baar-baar humari balcony mein aa jaati hai."}}, "gold": {"answer_en": "Luna", "slots": [{"name": "cat_name", "aliases": ["Luna", "लूना"]}]}, "distractor": {"value_aliases": ["Coco", "कोको"]}, "source": {"dataset": "LongMemEval", "question_id": "0862e8bf", "lme_type": "single-session-user"}, "topic": "pets", "is_answerable": true, "evidence_position": 2, "filler_seed": 3986903086, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F008", "cand_id": "FC151", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I visited Fushimi Inari Shrine when I was in Japan a few months ago. I spent two weeks travelling solo around the country. Can you recommend some budget-friendly places to stay in Kyoto?", "assistant_reply": "Kyoto has great budget options: guesthouses near Kyoto Station, capsule hotels downtown, and small ryokan-style hostels in the Gion area that still feel traditional.", "followup_turn": "Is it easy to get around Kyoto without a car?", "followup_reply": "Very easy. City buses reach most temples, the subway covers the centre, and renting a bicycle for a day is a popular and cheap way to explore.", "q1": "How long was I in Japan?", "q2": "How much time did I spend travelling in Japan?"}, "hi": {"memory_turn": "कुछ महीने पहले जब मैं जापान में था, तब मैं फ़ुशिमी इनारी श्राइन गया था। मैंने दो हफ़्ते अकेले पूरे देश में घूमते हुए बिताए। क्योटो में रुकने के लिए कुछ किफ़ायती जगहें बताएँगे?", "assistant_reply": "क्योटो में बजट के अच्छे विकल्प हैं: क्योटो स्टेशन के पास गेस्टहाउस, शहर के बीच कैप्सूल होटल, और गियोन इलाक़े में छोटे रयोकान-स्टाइल हॉस्टल, जिनमें पारंपरिक माहौल भी मिलता है।", "followup_turn": "क्या क्योटो में बिना कार के घूमना आसान है?", "followup_reply": "बहुत आसान है। शहर की बसें ज़्यादातर मंदिरों तक जाती हैं, सबवे शहर के बीच के हिस्से को कवर करता है, और एक दिन के लिए साइकिल किराए पर लेना घूमने का लोकप्रिय और सस्ता तरीक़ा है।", "q1": "मैं जापान में कितने समय तक रहा था?", "q2": "मैंने जापान घूमने में कितना समय बिताया?"}, "hg": {"memory_turn": "Kuch mahine pehle jab main Japan mein tha, tab main Fushimi Inari Shrine gaya tha. Maine do hafte akele poore country mein ghoomte hue bitaaye. Kyoto mein rukne ke liye kuch budget-friendly jagah batayenge?", "assistant_reply": "Kyoto mein budget ke acche options hain: Kyoto Station ke paas guesthouses, downtown mein capsule hotels, aur Gion area mein chhote ryokan-style hostels, jinmein traditional feel bhi milti hai.", "followup_turn": "Kya Kyoto mein bina car ke ghoomna easy hai?", "followup_reply": "Bahut easy hai. City buses zyadatar temples tak jaati hain, subway centre ko cover karta hai, aur ek din ke liye cycle rent karna ghoomne ka popular aur sasta tareeka hai.", "q1": "Main Japan mein kitne time tak raha tha?", "q2": "Maine Japan ghoomne mein kitna time bitaya?"}}, "gold": {"answer_en": "two weeks", "slots": [{"name": "duration", "aliases": ["two weeks", "2 weeks", "14 days", "fortnight", "दो हफ़्ते", "दो हफ्ते", "2 हफ़्ते", "दो सप्ताह", "14 दिन", "do hafte", "2 hafte", "14 din"]}]}, "source": {"dataset": "LongMemEval", "question_id": "19b5f2b3", "lme_type": "single-session-user"}, "topic": "travel", "is_answerable": true, "evidence_position": 5, "filler_seed": 4108670730, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F009", "cand_id": "FC155", "memory_type": "fact", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I just set up my new Samsung 55-inch 4K smart TV on Saturday and spent ages hiding the cables behind it. Can you recommend some good streaming services?", "assistant_reply": "Nice upgrade! Popular choices are Netflix for series, Amazon Prime Video for a big film collection, and JioHotstar for sports and Indian content.", "followup_turn": "Can I watch all of them through the TV's built-in apps?", "followup_reply": "Yes, Samsung smart TVs offer all three through the Tizen app store. You just need to download each app once and sign in with your account.", "q1": "What size is my new Samsung TV?", "q2": "How big is the new TV I set up?", "distractor_sentence": "My parents still use their old 43-inch TV in the living room."}, "hi": {"memory_turn": "मैंने शनिवार को अपना नया Samsung 55 इंच का 4K स्मार्ट टीवी सेट किया और उसके पीछे केबल छिपाने में काफ़ी समय लग गया। कोई अच्छी स्ट्रीमिंग सर्विस बताएँगे?", "assistant_reply": "बढ़िया अपग्रेड! सीरीज़ के लिए Netflix, फ़िल्मों के बड़े कलेक्शन के लिए Amazon Prime Video, और खेल व भारतीय कंटेंट के लिए JioHotstar लोकप्रिय विकल्प हैं।", "followup_turn": "क्या ये सब टीवी के बिल्ट-इन ऐप्स से चल जाएँगे?", "followup_reply": "हाँ, Samsung स्मार्ट टीवी पर ये तीनों Tizen ऐप स्टोर से मिल जाते हैं। बस हर ऐप एक बार डाउनलोड करके अपने अकाउंट से लॉग इन करना होगा।", "q1": "मेरा नया Samsung टीवी कितने इंच का है?", "q2": "मैंने जो नया टीवी लगाया है, वह कितना बड़ा है?", "distractor_sentence": "मेरे मम्मी-पापा अब भी लिविंग रूम में अपना पुराना 43 इंच वाला टीवी चलाते हैं।"}, "hg": {"memory_turn": "Maine Saturday ko apna naya Samsung 55-inch 4K smart TV set kiya aur uske peeche cables chhupane mein kaafi time lag gaya. Koi acchi streaming service batayenge?", "assistant_reply": "Badhiya upgrade! Series ke liye Netflix, movies ke bade collection ke liye Amazon Prime Video, aur sports aur Indian content ke liye JioHotstar popular options hain.", "followup_turn": "Kya yeh sab TV ke built-in apps se chal jayenge?", "followup_reply": "Haan, Samsung smart TV pe yeh teeno Tizen app store se mil jaate hain. Bas har app ek baar download karke apne account se login karna hoga.", "q1": "Mera naya Samsung TV kitne inch ka hai?", "q2": "Maine jo naya TV lagaya hai, woh kitna bada hai?", "distractor_sentence": "Mere mummy-papa ab bhi living room mein apna purana 43-inch wala TV chalate hain."}}, "gold": {"answer_en": "55 inches", "slots": [{"name": "size", "aliases": ["55-inch", "55 inch", "55 inches", "55 इंच", "55-इंच"]}]}, "distractor": {"value_aliases": ["43-inch", "43 inch", "43 inches", "43 इंच"]}, "source": {"dataset": "LongMemEval", "question_id": "76d63226", "lme_type": "single-session-user"}, "topic": "shopping/home", "is_answerable": true, "evidence_position": 9, "filler_seed": 1487936767, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F010", "cand_id": "FC161", "memory_type": "fact", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been practising guitar for 30 minutes every day and it's helping me progress nicely. Can you recommend some online resources for learning music theory and fingerpicking?", "assistant_reply": "Good habit! For theory, musictheory.net has free interactive lessons. For fingerpicking, JustinGuitar has a structured free course, and many teachers post step-by-step tutorials on YouTube.", "followup_turn": "Should I learn theory first or fingerpicking first?", "followup_reply": "Do both in parallel. Spend a few minutes on theory like scales and chord building, then apply it right away in a fingerpicking pattern so it sticks.", "q1": "How much time do I practise guitar every day?", "q2": "How long is my daily guitar practice?"}, "hi": {"memory_turn": "मैं रोज़ 30 मिनट गिटार की प्रैक्टिस कर रहा हूँ और इससे मेरी अच्छी प्रगति हो रही है। म्यूज़िक थ्योरी और फ़िंगरपिकिंग सीखने के लिए कुछ ऑनलाइन रिसोर्स बताएँगे?", "assistant_reply": "अच्छी आदत है! थ्योरी के लिए musictheory.net पर मुफ़्त इंटरैक्टिव लेसन हैं। फ़िंगरपिकिंग के लिए JustinGuitar पर एक व्यवस्थित मुफ़्त कोर्स है, और कई टीचर YouTube पर स्टेप-बाय-स्टेप ट्यूटोरियल डालते हैं।", "followup_turn": "पहले थ्योरी सीखूँ या फ़िंगरपिकिंग?", "followup_reply": "दोनों साथ-साथ सीखिए। कुछ मिनट स्केल और कॉर्ड बनाने जैसी थ्योरी पर लगाइए, फिर उसे तुरंत किसी फ़िंगरपिकिंग पैटर्न में आज़माइए ताकि वह याद रहे।", "q1": "मैं रोज़ कितनी देर गिटार की प्रैक्टिस करता हूँ?", "q2": "मेरी रोज़ की गिटार प्रैक्टिस कितने समय की होती है?"}, "hg": {"memory_turn": "Main roz 30 minute guitar practice kar raha hoon aur isse meri acchi progress ho rahi hai. Music theory aur fingerpicking seekhne ke liye kuch online resources batayenge?", "assistant_reply": "Acchi habit hai! Theory ke liye musictheory.net pe free interactive lessons hain. Fingerpicking ke liye JustinGuitar pe ek structured free course hai, aur kai teachers YouTube pe step-by-step tutorials daalte hain.", "followup_turn": "Pehle theory seekhun ya fingerpicking?", "followup_reply": "Dono saath-saath seekhiye. Kuch minute scales aur chord building jaisi theory pe lagaiye, phir use turant kisi fingerpicking pattern mein try kijiye taaki woh yaad rahe.", "q1": "Main roz kitni der guitar practice karta hoon?", "q2": "Meri roz ki guitar practice kitne time ki hoti hai?"}}, "gold": {"answer_en": "30 minutes", "slots": [{"name": "duration", "aliases": ["30 minutes", "30 minute", "30 mins", "30 min", "half an hour", "30 मिनट", "आधा घंटा", "aadha ghanta", "adha ghanta"]}]}, "source": {"dataset": "LongMemEval", "question_id": "29f2956b", "lme_type": "single-session-user"}, "topic": "hobbies", "is_answerable": true, "evidence_position": 2, "filler_seed": 2542195589, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P001", "cand_id": "FC125", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "Nike has been my favourite brand for running shoes so far. I'm looking for a new pair of the same model as my current ones. Where can I find them, or are there similar models?", "assistant_reply": "If the exact model is discontinued, the brand's website and big sports retailers often list a newer version under the same name with a new number. A running store can also match the fit.", "followup_turn": "How often should running shoes be replaced?", "followup_reply": "A common guideline is every 500 to 800 kilometres. Replace them sooner if the cushioning feels flat or the soles show uneven wear.", "q1": "Which brand of running shoes is my favourite?", "q2": "What's my favourite running shoe brand?", "distractor_sentence": "My brother swears by Asics for his runs, but I've never tried them."}, "hi": {"memory_turn": "अब तक रनिंग शूज़ के लिए Nike मेरा पसंदीदा ब्रांड रहा है। मैं अपने अभी वाले जूतों का ही मॉडल फिर से लेना चाहती हूँ। वे कहाँ मिलेंगे, या उनसे मिलते-जुलते मॉडल कौन-से हैं?", "assistant_reply": "अगर वही मॉडल बंद हो गया है, तो ब्रांड की वेबसाइट और बड़े स्पोर्ट्स स्टोर अक्सर उसी नाम से नए नंबर वाला वर्ज़न रखते हैं। रनिंग स्टोर पर फ़िट मिलाकर भी देख सकती हैं।", "followup_turn": "रनिंग शूज़ कितने समय बाद बदलने चाहिए?", "followup_reply": "आम तौर पर हर 500 से 800 किलोमीटर के बाद। अगर कुशनिंग दबी हुई लगे या तले असमान रूप से घिसने लगें, तो पहले भी बदल दें।", "q1": "रनिंग शूज़ का मेरा पसंदीदा ब्रांड कौन-सा है?", "q2": "दौड़ने वाले जूतों में मुझे कौन-सा ब्रांड सबसे ज़्यादा पसंद है?", "distractor_sentence": "मेरा भाई दौड़ने के लिए सिर्फ़ Asics पर भरोसा करता है, पर मैंने कभी नहीं पहने।"}, "hg": {"memory_turn": "Ab tak running shoes ke liye Nike mera favourite brand raha hai. Main apne abhi wale shoes ka hi model phir se lena chahti hoon. Woh kahan milenge, ya unse milte-julte models kaunse hain?", "assistant_reply": "Agar wahi model band ho gaya hai, toh brand ki website aur bade sports stores aksar usi naam se naye number wala version rakhte hain. Running store pe fit match karke bhi dekh sakti hain.", "followup_turn": "Running shoes kitne time baad badalne chahiye?", "followup_reply": "Usually har 500 se 800 kilometre ke baad. Agar cushioning dabi hui lage ya soles uneven ghisne lagein, toh pehle bhi badal dein.", "q1": "Running shoes ka mera favourite brand kaunsa hai?", "q2": "Daudne wale shoes mein mujhe kaunsa brand sabse zyada pasand hai?", "distractor_sentence": "Mera bhai running ke liye sirf Asics pe bharosa karta hai, par maine kabhi nahi pehne."}}, "gold": {"answer_en": "Nike", "slots": [{"name": "brand", "aliases": ["Nike", "नाइकी"]}]}, "distractor": {"value_aliases": ["Asics", "ASICS", "एसिक्स"]}, "source": {"dataset": "LongMemEval", "question_id": "c8c3f81d", "lme_type": "single-session-user"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 3743301637, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P002", "cand_id": "FC140", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm planning to make some Japanese-style dishes with my favourite rice, Japanese short-grain rice. Any simple recipes that pair well with it, maybe with grilled chicken or vegetables?", "assistant_reply": "Try chicken teriyaki rice bowls, a vegetable donburi with a soy and mirin glaze, or onigiri rice balls with a simple filling. All three take under 40 minutes.", "followup_turn": "How do I keep the rice from getting too sticky?", "followup_reply": "Rinse it several times until the water runs almost clear, use slightly less water than usual, and let it rest covered for ten minutes after cooking.", "q1": "What type of rice is my favourite?", "q2": "Which rice do I like best?"}, "hi": {"memory_turn": "मैं अपने पसंदीदा जापानी शॉर्ट-ग्रेन चावल से कुछ जापानी स्टाइल की डिशेज़ बनाने की सोच रहा हूँ। उसके साथ अच्छी लगने वाली कोई आसान रेसिपी बताएँगे, शायद ग्रिल्ड चिकन या सब्ज़ियों के साथ?", "assistant_reply": "चिकन तेरियाकी राइस बाउल, सोया और मिरिन ग्लेज़ वाला वेजिटेबल डोनबुरी, या आसान फ़िलिंग वाले ओनिगिरी राइस बॉल्स आज़माइए। तीनों 40 मिनट से कम में बन जाते हैं।", "followup_turn": "चावल को ज़्यादा चिपचिपा होने से कैसे बचाऊँ?", "followup_reply": "पानी लगभग साफ़ होने तक उसे कई बार धोइए, सामान्य से थोड़ा कम पानी डालिए, और पकने के बाद दस मिनट ढककर रहने दीजिए।", "q1": "मुझे कौन-सा चावल सबसे ज़्यादा पसंद है?", "q2": "मेरा पसंदीदा चावल किस तरह का है?"}, "hg": {"memory_turn": "Main apne favourite Japanese short-grain rice se kuch Japanese style dishes banane ki soch raha hoon. Uske saath acchi lagne wali koi easy recipe batayenge, shayad grilled chicken ya veggies ke saath?", "assistant_reply": "Chicken teriyaki rice bowl, soy aur mirin glaze wala vegetable donburi, ya simple filling wale onigiri rice balls try kijiye. Teeno 40 minute se kam mein ban jaate hain.", "followup_turn": "Rice ko zyada sticky hone se kaise bachaun?", "followup_reply": "Paani lagbhag saaf hone tak use kai baar dhoiye, normal se thoda kam paani daaliye, aur pakne ke baad das minute dhak kar rehne dijiye.", "q1": "Mujhe kaunsa rice sabse zyada pasand hai?", "q2": "Mera favourite rice kis type ka hai?"}}, "gold": {"answer_en": "Japanese short-grain rice", "slots": [{"name": "rice", "aliases": ["Japanese short-grain rice", "Japanese short grain", "short-grain", "short grain", "Japanese rice", "जापानी शॉर्ट-ग्रेन", "शॉर्ट-ग्रेन", "जापानी चावल"]}]}, "source": {"dataset": "LongMemEval", "question_id": "60d45044", "lme_type": "single-session-user"}, "topic": "food/cooking", "is_answerable": true, "evidence_position": 5, "filler_seed": 4003791473, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P003", "cand_id": "PC075", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm trying to learn some advanced settings for video editing in Adobe Premiere Pro, which I really enjoy using. Can you give me some tips on where to start?", "assistant_reply": "Start with three areas: proxy workflows for smooth playback, the colour grading panels, and customising keyboard shortcuts. Mastering these speeds up almost every project.", "followup_turn": "What is a proxy workflow exactly?", "followup_reply": "You create lightweight copies of your high-resolution footage, edit with those for smooth playback, and the software switches back to the original files when you export.", "q1": "Which video editing software do I enjoy using?", "q2": "What's my favourite tool for editing videos?"}, "hi": {"memory_turn": "मैं Adobe Premiere Pro में वीडियो एडिटिंग की कुछ एडवांस्ड सेटिंग्स सीखने की कोशिश कर रही हूँ, मुझे इसे इस्तेमाल करना बहुत पसंद है। शुरुआत कहाँ से करूँ, कुछ टिप्स देंगे?", "assistant_reply": "तीन चीज़ों से शुरू कीजिए: स्मूद प्लेबैक के लिए प्रॉक्सी वर्कफ़्लो, कलर ग्रेडिंग पैनल, और कीबोर्ड शॉर्टकट को अपने हिसाब से सेट करना। इनमें माहिर होने से लगभग हर प्रोजेक्ट तेज़ हो जाता है।", "followup_turn": "प्रॉक्सी वर्कफ़्लो असल में क्या होता है?", "followup_reply": "आप अपनी हाई-रिज़ॉल्यूशन फ़ुटेज की हल्की कॉपियाँ बनाते हैं, स्मूद प्लेबैक के लिए उन्हीं से एडिट करते हैं, और एक्सपोर्ट के समय सॉफ़्टवेयर फिर से ओरिजिनल फ़ाइलें इस्तेमाल करता है।", "q1": "मुझे वीडियो एडिट करने के लिए कौन-सा सॉफ़्टवेयर इस्तेमाल करना पसंद है?", "q2": "वीडियो एडिटिंग के लिए मेरा पसंदीदा टूल कौन-सा है?"}, "hg": {"memory_turn": "Main Adobe Premiere Pro mein video editing ki kuch advanced settings seekhne ki koshish kar rahi hoon, mujhe ise use karna bahut pasand hai. Start kahan se karoon, kuch tips denge?", "assistant_reply": "Teen cheezon se start kijiye: smooth playback ke liye proxy workflow, colour grading panels, aur keyboard shortcuts ko apne hisaab se set karna. Inmein expert hone se lagbhag har project fast ho jaata hai.", "followup_turn": "Proxy workflow actually kya hota hai?", "followup_reply": "Aap apni high-resolution footage ki light copies banate hain, smooth playback ke liye unhi se edit karte hain, aur export ke time software phir se original files use karta hai.", "q1": "Mujhe video edit karne ke liye kaunsa software use karna pasand hai?", "q2": "Video editing ke liye mera favourite tool kaunsa hai?"}}, "gold": {"answer_en": "Adobe Premiere Pro", "slots": [{"name": "software", "aliases": ["Adobe Premiere Pro", "Premiere Pro", "Premiere", "एडोबी प्रीमियर प्रो", "प्रीमियर प्रो"]}]}, "source": {"dataset": "LongMemEval", "question_id": "8a2466db", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 302390493, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P004", "cand_id": "PC076", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm looking to upgrade my camera flash. Can you recommend some good options that are compatible with my Sony A7R IV?", "assistant_reply": "Good options include the Godox V1 for its round head and fast recycling, the Profoto A10 if budget allows, and the Godox TT685 II as a reliable value pick. Make sure you buy the version for your mount.", "followup_turn": "Do I need TTL support, or is manual flash enough?", "followup_reply": "TTL is very helpful at events where light changes quickly, because the flash meters automatically. For studio or planned shots, manual control gives more consistent results.", "q1": "Which camera brand do I use?", "q2": "What brand is my camera?"}, "hi": {"memory_turn": "मैं अपना कैमरा फ़्लैश अपग्रेड करना चाहता हूँ। मेरे Sony A7R IV के साथ चलने वाले कुछ अच्छे विकल्प बताएँगे?", "assistant_reply": "अच्छे विकल्प हैं: गोल हेड और तेज़ रीसाइक्लिंग के लिए Godox V1, बजट हो तो Profoto A10, और भरोसेमंद किफ़ायती विकल्प के तौर पर Godox TT685 II। ख़रीदते समय अपने माउंट वाला वर्ज़न ज़रूर देखें।", "followup_turn": "क्या मुझे TTL सपोर्ट चाहिए, या मैनुअल फ़्लैश काफ़ी है?", "followup_reply": "जहाँ रोशनी जल्दी-जल्दी बदलती है, जैसे इवेंट्स में, वहाँ TTL बहुत काम आता है क्योंकि फ़्लैश ख़ुद रोशनी नापता है। स्टूडियो या प्लान किए गए शॉट्स में मैनुअल कंट्रोल से ज़्यादा एक जैसे नतीजे मिलते हैं।", "q1": "मैं किस ब्रांड का कैमरा इस्तेमाल करता हूँ?", "q2": "मेरा कैमरा किस कंपनी का है?"}, "hg": {"memory_turn": "Main apna camera flash upgrade karna chahta hoon. Mere Sony A7R IV ke saath chalne wale kuch acche options batayenge?", "assistant_reply": "Acche options hain: round head aur fast recycling ke liye Godox V1, budget ho toh Profoto A10, aur reliable value pick ke taur pe Godox TT685 II. Khareedte time apne mount wala version zaroor dekhein.", "followup_turn": "Kya mujhe TTL support chahiye, ya manual flash kaafi hai?", "followup_reply": "Jahan light jaldi-jaldi badalti hai, jaise events mein, wahan TTL bahut kaam aata hai kyunki flash khud light measure karta hai. Studio ya planned shots mein manual control se zyada consistent results milte hain.", "q1": "Main kis brand ka camera use karta hoon?", "q2": "Mera camera kis company ka hai?"}}, "gold": {"answer_en": "Sony (A7R IV)", "slots": [{"name": "brand", "aliases": ["Sony", "सोनी"]}]}, "source": {"dataset": "LongMemEval", "question_id": "06878be2", "lme_type": "single-session-preference"}, "topic": "hobbies", "is_answerable": true, "evidence_position": 2, "filler_seed": 12467665, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P005", "cand_id": "PC081", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "What else can I do in the later part of the day? I prefer winding down by 9:30 pm so I can get a good night's sleep.", "assistant_reply": "Some calm options: a short walk after dinner, light stretching, journaling, or reading a paper book. Try to keep screens away for the last half hour before you wind down.", "followup_turn": "Does herbal tea actually help with sleep?", "followup_reply": "For many people, yes. Caffeine-free teas like chamomile can feel relaxing as part of a routine, though the habit itself often matters as much as the tea.", "q1": "By what time do I prefer to wind down at night?", "q2": "What time do I like to start winding down for sleep?"}, "hi": {"memory_turn": "दिन के आख़िरी हिस्से में मैं और क्या कर सकती हूँ? मुझे रात 9:30 बजे तक आराम के मूड में आ जाना पसंद है, ताकि अच्छी नींद आ सके।", "assistant_reply": "कुछ शांत विकल्प: खाने के बाद थोड़ी सैर, हल्की स्ट्रेचिंग, डायरी लिखना, या कोई किताब पढ़ना। सोने की तैयारी से पहले के आधे घंटे में स्क्रीन से दूर रहने की कोशिश करें।", "followup_turn": "क्या हर्बल चाय सच में नींद में मदद करती है?", "followup_reply": "कई लोगों के लिए हाँ। कैमोमाइल जैसी बिना कैफ़ीन वाली चाय रूटीन का हिस्सा बनकर सुकून देती है, हालाँकि अक्सर आदत उतनी ही मायने रखती है जितनी चाय।", "q1": "रात को मैं कितने बजे तक आराम के मूड में आ जाना पसंद करती हूँ?", "q2": "सोने से पहले मुझे किस समय तक रिलैक्स होना शुरू कर देना पसंद है?"}, "hg": {"memory_turn": "Din ke last part mein main aur kya kar sakti hoon? Mujhe raat 9:30 baje tak wind down kar lena pasand hai, taaki acchi neend aa sake.", "assistant_reply": "Kuch calm options: dinner ke baad thodi walk, light stretching, journaling, ya koi book padhna. Wind down se pehle ke aadhe ghante mein screens se door rehne ki koshish karein.", "followup_turn": "Kya herbal tea sach mein neend mein help karti hai?", "followup_reply": "Kai logon ke liye haan. Chamomile jaisi caffeine-free tea routine ka hissa bankar relax feel karati hai, halaanki aksar habit utni hi matter karti hai jitni tea.", "q1": "Raat ko main kitne baje tak wind down karna pasand karti hoon?", "q2": "Sone se pehle mujhe kis time tak relax hona shuru kar dena pasand hai?"}}, "gold": {"answer_en": "9:30 pm", "slots": [{"name": "time", "aliases": ["9:30", "9.30", "half past nine", "साढ़े नौ", "sadhe nau", "saade nau"]}]}, "source": {"dataset": "LongMemEval", "question_id": "195a1a1b", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 3815814950, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P006", "cand_id": "PC085", "memory_type": "preference", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I've been using basil and mint in my cooking a lot lately, and I've even harvested some cherry tomatoes from my garden. Which companion plants could help my tomatoes grow better?", "assistant_reply": "Marigolds are a classic companion because they help deter some pests. Chives and garlic planted nearby can also help, while borage attracts pollinators to the tomato flowers.", "followup_turn": "How often should I water the tomatoes in hot weather?", "followup_reply": "Deep watering every two to three days is usually better than a little every day. Water at the base in the morning so the leaves stay dry.", "q1": "Which herbs have I been using in my cooking lately?", "q2": "What herbs do I like to cook with these days?", "distractor_sentence": "My mom, on the other hand, mostly cooks with coriander and curry leaves."}, "hi": {"memory_turn": "आजकल मैं अपने खाने में बेसिल और पुदीना बहुत इस्तेमाल कर रहा हूँ, और मैंने अपने बग़ीचे से कुछ चेरी टमाटर भी तोड़े हैं। टमाटर बेहतर उगें, इसके लिए साथ में कौन-से पौधे लगाऊँ?", "assistant_reply": "गेंदा एक पुराना और अच्छा साथी पौधा है, क्योंकि यह कुछ कीड़ों को दूर रखता है। पास में चाइव्स और लहसुन लगाने से भी फ़ायदा होता है, और बोरेज टमाटर के फूलों तक परागण करने वाले कीड़ों को खींचता है।", "followup_turn": "गर्मी में टमाटर को कितनी बार पानी देना चाहिए?", "followup_reply": "रोज़ थोड़ा-थोड़ा देने के बजाय हर दो-तीन दिन में अच्छी तरह गहरा पानी देना बेहतर है। सुबह जड़ के पास पानी दें ताकि पत्तियाँ सूखी रहें।", "q1": "आजकल मैं अपने खाने में कौन-सी जड़ी-बूटियाँ इस्तेमाल कर रहा हूँ?", "q2": "इन दिनों मुझे किन हर्ब्स के साथ खाना बनाना पसंद है?", "distractor_sentence": "वहीं मेरी मम्मी ज़्यादातर धनिया और करी पत्ते से खाना बनाती हैं।"}, "hg": {"memory_turn": "Aajkal main apne khane mein basil aur pudina bahut use kar raha hoon, aur maine apne garden se kuch cherry tomatoes bhi tode hain. Tomatoes better ugein, iske liye saath mein kaunse plants lagaun?", "assistant_reply": "Marigold ek classic companion plant hai, kyunki yeh kuch keedon ko door rakhta hai. Paas mein chives aur lehsun lagane se bhi fayda hota hai, aur borage tomato ke phoolon tak pollinators ko kheenchta hai.", "followup_turn": "Garmi mein tomatoes ko kitni baar paani dena chahiye?", "followup_reply": "Roz thoda-thoda dene ke bajaye har do-teen din mein acche se deep watering better hai. Subah jadh ke paas paani dein taaki patte sookhe rahein.", "q1": "Aajkal main apne khane mein kaunse herbs use kar raha hoon?", "q2": "In dino mujhe kin herbs ke saath khana banana pasand hai?", "distractor_sentence": "Wahin meri mummy zyadatar dhaniya aur kadi patta se khana banati hain."}}, "gold": {"answer_en": "basil and mint", "slots": [{"name": "herb_1", "aliases": ["basil", "बेसिल", "तुलसी"]}, {"name": "herb_2", "aliases": ["mint", "पुदीना", "pudina", "मिंट"]}]}, "distractor": {"value_aliases": ["coriander", "धनिया", "dhaniya", "curry leaves", "करी पत्ते", "करी पत्ता", "kadi patta"]}, "source": {"dataset": "LongMemEval", "question_id": "06f04340", "lme_type": "single-session-preference"}, "topic": "hobbies", "is_answerable": true, "evidence_position": 9, "filler_seed": 2107664876, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P007", "cand_id": "PC091", "memory_type": "preference", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm looking for some mid-century modern design inspiration for a new bedroom dresser. Do you have any recommendations for websites or designers I should check out?", "assistant_reply": "Browse interior design websites and Pinterest boards, look through furniture stores' catalogues, and check second-hand marketplaces for original vintage pieces at good prices.", "followup_turn": "Would walnut or teak look better in a small bedroom?", "followup_reply": "Teak is slightly lighter and warmer, so it can make a small room feel more open. Walnut is darker and richer, which works well if your walls are light.", "q1": "Which furniture design style do I like?", "q2": "What design style am I looking at for my bedroom furniture?", "distractor_sentence": "My roommate is all about Scandinavian minimalism, though."}, "hi": {"memory_turn": "मैं अपने बेडरूम के नए ड्रेसर के लिए मिड-सेंचुरी मॉडर्न डिज़ाइन से कुछ प्रेरणा ढूँढ रही हूँ। कोई वेबसाइट या डिज़ाइनर बताएँगे जिन्हें मैं देखूँ?", "assistant_reply": "इंटीरियर डिज़ाइन की वेबसाइट और Pinterest बोर्ड देखिए, फ़र्नीचर दुकानों के कैटलॉग पलटिए, और अच्छे दाम पर पुराने विंटेज पीस के लिए सेकंड-हैंड मार्केटप्लेस भी देखिए।", "followup_turn": "छोटे बेडरूम में अखरोट की लकड़ी अच्छी लगेगी या सागौन?", "followup_reply": "सागौन थोड़ा हल्का और गर्म रंग का होता है, इसलिए छोटा कमरा ज़्यादा खुला लगता है। अखरोट गहरा और भरपूर रंग का होता है, जो हल्की दीवारों के साथ अच्छा लगता है।", "q1": "मुझे फ़र्नीचर का कौन-सा डिज़ाइन स्टाइल पसंद है?", "q2": "अपने बेडरूम के फ़र्नीचर के लिए मैं कौन-सा डिज़ाइन स्टाइल देख रही हूँ?", "distractor_sentence": "हालाँकि मेरी रूममेट को स्कैंडिनेवियन मिनिमलिज़्म ही पसंद है।"}, "hg": {"memory_turn": "Main apne bedroom ke naye dresser ke liye mid-century modern design se kuch inspiration dhoondh rahi hoon. Koi websites ya designers batayenge jinhe main dekhoon?", "assistant_reply": "Interior design websites aur Pinterest boards dekhiye, furniture stores ke catalogues check kijiye, aur acche price pe original vintage pieces ke liye second-hand marketplaces bhi dekhiye.", "followup_turn": "Chhote bedroom mein walnut accha lagega ya teak?", "followup_reply": "Teak thoda light aur warm hota hai, isliye chhota room zyada open lagta hai. Walnut dark aur rich hota hai, jo light walls ke saath accha lagta hai.", "q1": "Mujhe furniture ka kaunsa design style pasand hai?", "q2": "Apne bedroom furniture ke liye main kaunsa design style dekh rahi hoon?", "distractor_sentence": "Halaanki meri roommate ko Scandinavian minimalism hi pasand hai."}}, "gold": {"answer_en": "mid-century modern", "slots": [{"name": "style", "aliases": ["mid-century modern", "mid century modern", "mid-century", "मिड-सेंचुरी मॉडर्न", "मिड सेंचुरी मॉडर्न", "मिड-सेंचुरी"]}]}, "distractor": {"value_aliases": ["Scandinavian", "minimalism", "स्कैंडिनेवियन", "मिनिमलिज़्म"]}, "source": {"dataset": "LongMemEval", "question_id": "57f827a0", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 1949295071, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P008", "cand_id": "PC092", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I'm considering upgrading from my Fender Stratocaster to a Gibson Les Paul. Can you tell me the main differences between these two guitars?", "assistant_reply": "The biggest differences are sound and feel. One has single-coil pickups with a bright, clear tone and a lighter body; the other has humbuckers with a thicker, warmer sound and a heavier body.", "followup_turn": "Is the heavier weight a problem for long gigs?", "followup_reply": "It can be. Many players use a wide padded strap to spread the load, and some models have chambered bodies that weigh noticeably less.", "q1": "Which guitar am I thinking of upgrading to?", "q2": "What guitar do I want to switch to?"}, "hi": {"memory_turn": "मैं अपने Fender Stratocaster से Gibson Les Paul पर अपग्रेड करने की सोच रहा हूँ। इन दोनों गिटार में मुख्य फ़र्क़ क्या हैं?", "assistant_reply": "सबसे बड़ा फ़र्क़ आवाज़ और पकड़ में है। एक में सिंगल-कॉइल पिकअप होते हैं, जिनकी आवाज़ तीखी और साफ़ होती है और बॉडी हल्की होती है; दूसरे में हम्बकर होते हैं, जिनकी आवाज़ भारी और गर्म होती है और बॉडी वज़नदार होती है।", "followup_turn": "क्या लंबे शो में ज़्यादा वज़न परेशानी बनता है?", "followup_reply": "बन सकता है। कई गिटारिस्ट वज़न बाँटने के लिए चौड़ा गद्देदार स्ट्रैप लगाते हैं, और कुछ मॉडल की बॉडी अंदर से खोखली होती है जिससे वे काफ़ी हल्के होते हैं।", "q1": "मैं किस गिटार पर अपग्रेड करने की सोच रहा हूँ?", "q2": "मैं अब कौन-सा गिटार लेना चाहता हूँ?"}, "hg": {"memory_turn": "Main apne Fender Stratocaster se Gibson Les Paul pe upgrade karne ki soch raha hoon. In dono guitars mein main differences kya hain?", "assistant_reply": "Sabse bada difference sound aur feel mein hai. Ek mein single-coil pickups hote hain, jinki sound bright aur clear hoti hai aur body light hoti hai; doosre mein humbuckers hote hain, jinki sound thick aur warm hoti hai aur body heavy hoti hai.", "followup_turn": "Kya lambe gigs mein zyada weight problem banta hai?", "followup_reply": "Ban sakta hai. Kai players weight distribute karne ke liye wide padded strap use karte hain, aur kuch models ki body andar se chambered hoti hai jisse woh kaafi light hote hain.", "q1": "Main kis guitar pe upgrade karne ki soch raha hoon?", "q2": "Main ab kaunsa guitar lena chahta hoon?"}}, "gold": {"answer_en": "Gibson Les Paul", "slots": [{"name": "guitar", "aliases": ["Les Paul", "लेस पॉल"]}]}, "near_miss_aliases": ["Stratocaster", "Strat", "स्ट्रैटोकास्टर"], "source": {"dataset": "LongMemEval", "question_id": "95228167", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 3781988423, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P009", "cand_id": "PC093", "memory_type": "preference", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I'm trying to cut down on sugar and save money, so I've started making my own flavoured creamer with almond milk, vanilla extract and honey. Any tips to make it even healthier?", "assistant_reply": "You could use a little less honey and add cinnamon or cardamom for sweetness without sugar. A pinch of salt also boosts flavour, and a glass jar keeps it fresh for about five days.", "followup_turn": "Can I make a chocolate version too?", "followup_reply": "Yes, whisk in a spoonful of unsweetened cocoa powder while it is slightly warm. Shake the jar before each use because cocoa tends to settle.", "q1": "Which milk do I use for my homemade creamer?", "q2": "What kind of milk goes into the creamer I make at home?"}, "hi": {"memory_turn": "मैं चीनी कम करना और पैसे बचाना चाहती हूँ, इसलिए मैंने बादाम के दूध, वनीला एक्सट्रैक्ट और शहद से अपना फ़्लेवर्ड क्रीमर ख़ुद बनाना शुरू किया है। इसे और हेल्दी बनाने के लिए कुछ टिप्स देंगे?", "assistant_reply": "शहद थोड़ा कम करके मिठास के लिए दालचीनी या इलायची डाल सकती हैं। एक चुटकी नमक से स्वाद बढ़ता है, और काँच के जार में रखने से यह लगभग पाँच दिन ताज़ा रहता है।", "followup_turn": "क्या मैं इसका चॉकलेट वाला वर्ज़न भी बना सकती हूँ?", "followup_reply": "हाँ, हल्का गुनगुना रहते हुए एक चम्मच बिना चीनी वाला कोको पाउडर फेंट दीजिए। हर बार इस्तेमाल से पहले जार हिला लें, क्योंकि कोको नीचे बैठ जाता है।", "q1": "अपना घर का क्रीमर बनाने के लिए मैं कौन-सा दूध इस्तेमाल करती हूँ?", "q2": "घर पर जो क्रीमर मैं बनाती हूँ, उसमें किस तरह का दूध डलता है?"}, "hg": {"memory_turn": "Main sugar kam karna aur paise bachana chahti hoon, isliye maine almond milk, vanilla extract aur honey se apna flavoured creamer khud banana shuru kiya hai. Ise aur healthy banane ke liye kuch tips denge?", "assistant_reply": "Honey thoda kam karke sweetness ke liye cinnamon ya elaichi daal sakti hain. Ek chutki namak se taste badhta hai, aur glass jar mein rakhne se yeh lagbhag paanch din fresh rehta hai.", "followup_turn": "Kya main iska chocolate wala version bhi bana sakti hoon?", "followup_reply": "Haan, halka garam rehte hue ek chammach unsweetened cocoa powder mila dijiye. Har baar use karne se pehle jar hila lein, kyunki cocoa neeche baith jaata hai.", "q1": "Apna ghar ka creamer banane ke liye main kaunsa milk use karti hoon?", "q2": "Ghar pe jo creamer main banati hoon, usmein kis type ka milk jaata hai?"}}, "gold": {"answer_en": "almond milk", "slots": [{"name": "milk", "aliases": ["almond milk", "almond", "बादाम के दूध", "बादाम का दूध", "बादाम दूध", "आमंड मिल्क", "badam ka doodh", "badam milk"]}]}, "source": {"dataset": "LongMemEval", "question_id": "505af2f5", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 683709250, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P010", "cand_id": "PC100", "memory_type": "preference", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I've been watching a lot of documentaries lately, mostly on Netflix. Can you recommend more documentary series like Our Planet and Free Solo, which I just finished?", "assistant_reply": "If you liked those, try Planet Earth II for stunning nature footage, Meru for another climbing story, and Blue Planet II for ocean life.", "followup_turn": "Is Meru as intense as Free Solo?", "followup_reply": "In a different way. Meru follows a team on a dangerous climb over several years, so it is more about persistence and friendship than one single climb.", "q1": "Where do I usually watch documentaries?", "q2": "Which streaming service do I mostly use for documentaries?"}, "hi": {"memory_turn": "आजकल मैं बहुत सारी डॉक्यूमेंट्री देख रहा हूँ, ज़्यादातर Netflix पर। Our Planet और Free Solo जैसी कुछ और डॉक्यूमेंट्री सीरीज़ बताएँगे? ये दोनों मैंने अभी-अभी ख़त्म की हैं।", "assistant_reply": "अगर वे पसंद आईं, तो शानदार प्रकृति के दृश्यों के लिए Planet Earth II, चढ़ाई पर बनी फ़िल्म Meru, और समुद्री जीवन के लिए Blue Planet II देखिए।", "followup_turn": "क्या Meru भी Free Solo जितनी रोमांचक है?", "followup_reply": "अलग तरह से। Meru में कुछ पर्वतारोहियों की कई साल लंबी ख़तरनाक चढ़ाई दिखाई गई है, इसलिए यह एक चढ़ाई से ज़्यादा लगन और दोस्ती की कहानी है।", "q1": "मैं आमतौर पर डॉक्यूमेंट्री कहाँ देखता हूँ?", "q2": "डॉक्यूमेंट्री देखने के लिए मैं ज़्यादातर कौन-सी स्ट्रीमिंग सर्विस इस्तेमाल करता हूँ?"}, "hg": {"memory_turn": "Aajkal main bahut saari documentaries dekh raha hoon, zyadatar Netflix pe. Our Planet aur Free Solo jaisi kuch aur documentary series batayenge? Yeh dono maine abhi-abhi khatam ki hain.", "assistant_reply": "Agar woh pasand aayi, toh amazing nature visuals ke liye Planet Earth II, climbing pe bani film Meru, aur ocean life ke liye Blue Planet II dekhiye.", "followup_turn": "Kya Meru bhi Free Solo jitni thrilling hai?", "followup_reply": "Alag tarah se. Meru mein kuch climbers ki kai saal lambi khatarnaak chadhai dikhayi gayi hai, isliye yeh ek climb se zyada persistence aur dosti ki kahani hai.", "q1": "Main usually documentaries kahan dekhta hoon?", "q2": "Documentaries dekhne ke liye main zyadatar kaunsi streaming service use karta hoon?"}}, "gold": {"answer_en": "Netflix", "slots": [{"name": "service", "aliases": ["Netflix", "नेटफ़्लिक्स", "नेटफ्लिक्स"]}]}, "source": {"dataset": "LongMemEval", "question_id": "b0479f84", "lme_type": "single-session-preference"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 580460840, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E001", "cand_id": "EC001", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I'm thinking of getting my car detailed soon. Any tips on choosing a good detailer? By the way, I got my car serviced for the first time on March 15th, and it was a great experience.", "assistant_reply": "Glad the service went well! For detailing, check recent reviews with photos, ask exactly what each package includes, and confirm they use pH-neutral products that are safe for your paint.", "followup_turn": "Is ceramic coating worth paying extra for?", "followup_reply": "It can be if you keep the car for several years. It makes washing easier and protects the paint, but it is not scratch-proof and needs proper preparation to last.", "q1": "When did I get my car serviced for the first time?", "q2": "On what date was my car's first service?", "distractor_sentence": "My dad got his car serviced on April 2nd, and it took the whole day."}, "hi": {"memory_turn": "मैं जल्द ही अपनी कार की डिटेलिंग करवाने की सोच रही हूँ। अच्छा डिटेलर चुनने के लिए कोई टिप्स? वैसे, मैंने 15 मार्च को पहली बार अपनी कार की सर्विस करवाई थी, और अनुभव बहुत अच्छा रहा।", "assistant_reply": "अच्छा लगा कि सर्विस ठीक रही! डिटेलिंग के लिए फ़ोटो वाले हाल के रिव्यू देखिए, हर पैकेज में ठीक-ठीक क्या शामिल है यह पूछिए, और पक्का कीजिए कि वे पेंट के लिए सुरक्षित pH-न्यूट्रल प्रोडक्ट इस्तेमाल करते हैं।", "followup_turn": "क्या सिरेमिक कोटिंग के लिए ज़्यादा पैसे देना ठीक है?", "followup_reply": "अगर आप कार कई साल रखेंगी तो हाँ। इससे धोना आसान होता है और पेंट सुरक्षित रहता है, लेकिन यह स्क्रैच-प्रूफ़ नहीं है और टिकाऊ होने के लिए सही तैयारी ज़रूरी है।", "q1": "मैंने पहली बार अपनी कार की सर्विस कब करवाई थी?", "q2": "मेरी कार की पहली सर्विस किस तारीख़ को हुई थी?", "distractor_sentence": "मेरे पापा ने 2 अप्रैल को अपनी कार की सर्विस करवाई थी, और पूरा दिन लग गया था।"}, "hg": {"memory_turn": "Main jaldi hi apni car ki detailing karwane ki soch rahi hoon. Accha detailer choose karne ke liye koi tips? Waise, maine 15 March ko pehli baar apni car ki service karwayi thi, aur experience bahut accha raha.", "assistant_reply": "Accha laga ki service theek rahi! Detailing ke liye photos wale recent reviews dekhiye, har package mein exactly kya include hai yeh poochhiye, aur confirm kijiye ki woh paint ke liye safe pH-neutral products use karte hain.", "followup_turn": "Kya ceramic coating ke liye extra paise dena worth it hai?", "followup_reply": "Agar aap car kai saal rakhengi toh haan. Isse washing easy hoti hai aur paint protected rehta hai, lekin yeh scratch-proof nahi hai aur tikne ke liye sahi preparation zaroori hai.", "q1": "Maine pehli baar apni car ki service kab karwayi thi?", "q2": "Meri car ki pehli service kis date ko hui thi?", "distractor_sentence": "Mere papa ne 2 April ko apni car ki service karwayi thi, aur poora din lag gaya tha."}}, "gold": {"answer_en": "March 15", "date": {"day": 15, "month": 3}, "slots": [{"name": "date", "aliases": ["March 15", "March 15th", "15 March", "15th March", "15th of March", "Mar 15", "15 Mar", "15 मार्च"]}]}, "distractor": {"date": {"day": 2, "month": 4}, "value_aliases": ["April 2", "April 2nd", "2 April", "2nd April", "2nd of April", "Apr 2", "2 Apr", "2 अप्रैल"]}, "source": {"dataset": "LongMemEval", "question_id": "gpt4_2655b836", "lme_type": "temporal-reasoning"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 2, "filler_seed": 2791523688, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E002", "cand_id": "EC003", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I got a new Samsung Galaxy S22 from the electronics store at the mall on February 20th, and I'm loving it so far. What are some must-have settings I should turn on?", "assistant_reply": "A few useful ones: enable the adaptive refresh rate for smoother scrolling, set up Find My Mobile in case it gets lost, and turn on battery protection to limit charging to 85 percent.", "followup_turn": "Should I get a case with a built-in screen protector?", "followup_reply": "It is a good choice if you want full protection in one product. If you prefer a slimmer phone, a separate tempered glass protector with a thin case works just as well.", "q1": "When did I get my Samsung Galaxy S22?", "q2": "On what date did I buy my new Galaxy S22?"}, "hi": {"memory_turn": "मैंने 20 फ़रवरी को मॉल के इलेक्ट्रॉनिक्स स्टोर से नया Samsung Galaxy S22 लिया, और अब तक मुझे यह बहुत पसंद आ रहा है। कौन-सी सेटिंग्स ज़रूर ऑन करनी चाहिए?", "assistant_reply": "कुछ काम की सेटिंग्स: स्मूद स्क्रॉलिंग के लिए अडैप्टिव रिफ़्रेश रेट ऑन करें, खो जाने की स्थिति के लिए Find My Mobile सेट करें, और चार्जिंग को 85 प्रतिशत तक सीमित करने के लिए बैटरी प्रोटेक्शन ऑन करें।", "followup_turn": "क्या बिल्ट-इन स्क्रीन प्रोटेक्टर वाला कवर लेना चाहिए?", "followup_reply": "अगर एक ही चीज़ में पूरी सुरक्षा चाहिए तो यह अच्छा विकल्प है। अगर फ़ोन पतला रखना पसंद है, तो अलग टेम्पर्ड ग्लास के साथ पतला कवर भी उतना ही अच्छा काम करता है।", "q1": "मैंने अपना Samsung Galaxy S22 कब लिया था?", "q2": "मैंने नया Galaxy S22 किस तारीख़ को ख़रीदा था?"}, "hg": {"memory_turn": "Maine 20 February ko mall ke electronics store se naya Samsung Galaxy S22 liya, aur ab tak mujhe yeh bahut pasand aa raha hai. Kaunsi settings zaroor on karni chahiye?", "assistant_reply": "Kuch kaam ki settings: smooth scrolling ke liye adaptive refresh rate on karein, kho jaane ki situation ke liye Find My Mobile set karein, aur charging ko 85 percent tak limit karne ke liye battery protection on karein.", "followup_turn": "Kya built-in screen protector wala cover lena chahiye?", "followup_reply": "Agar ek hi cheez mein full protection chahiye toh yeh accha option hai. Agar phone slim rakhna pasand hai, toh alag tempered glass ke saath patla cover bhi utna hi accha kaam karta hai.", "q1": "Maine apna Samsung Galaxy S22 kab liya tha?", "q2": "Maine naya Galaxy S22 kis date ko khareeda tha?"}}, "gold": {"answer_en": "February 20", "date": {"day": 20, "month": 2}, "slots": [{"name": "date", "aliases": ["February 20", "February 20th", "20 February", "20th February", "20th of February", "Feb 20", "20 Feb", "20 फ़रवरी", "20 फरवरी"]}]}, "source": {"dataset": "LongMemEval", "question_id": "gpt4_2312f94c", "lme_type": "temporal-reasoning"}, "topic": "shopping/home", "is_answerable": true, "evidence_position": 5, "filler_seed": 1908762337, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E003", "cand_id": "EC005", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I attended a workshop on Effective Communication in the Workplace on January 10th, and it really helped my active listening and conflict resolution skills. How can I keep practising these at work?", "assistant_reply": "Try summarising what a colleague said before you respond, ask one clarifying question in each meeting, and when a disagreement starts, name the shared goal before discussing solutions.", "followup_turn": "What if a colleague keeps interrupting me in meetings?", "followup_reply": "Stay calm and say something like: let me finish this point, then I would love to hear yours. If it continues, raise it with them privately rather than in front of the team.", "q1": "When did I attend the workshop on Effective Communication in the Workplace?", "q2": "On what date was the communication workshop I went to?"}, "hi": {"memory_turn": "मैंने 10 जनवरी को 'कार्यस्थल पर प्रभावी संवाद' पर एक वर्कशॉप में हिस्सा लिया, और इससे ध्यान से सुनने और विवाद सुलझाने की मेरी स्किल्स में बहुत मदद मिली। ऑफ़िस में इनकी प्रैक्टिस कैसे जारी रखूँ?", "assistant_reply": "जवाब देने से पहले सहकर्मी की बात को संक्षेप में दोहराइए, हर मीटिंग में कम से कम एक स्पष्ट करने वाला सवाल पूछिए, और जब असहमति शुरू हो तो समाधान पर बात करने से पहले साझा लक्ष्य बताइए।", "followup_turn": "अगर कोई सहकर्मी मीटिंग में बार-बार मुझे बीच में टोके तो क्या करूँ?", "followup_reply": "शांत रहिए और कुछ ऐसा कहिए: मुझे यह बात पूरी करने दीजिए, फिर मैं आपकी बात ज़रूर सुनना चाहूँगी। अगर यह जारी रहे, तो टीम के सामने नहीं बल्कि अकेले में उनसे बात कीजिए।", "q1": "मैं 'कार्यस्थल पर प्रभावी संवाद' वाली वर्कशॉप में कब गई थी?", "q2": "जिस कम्युनिकेशन वर्कशॉप में मैं गई थी, वह किस तारीख़ को थी?"}, "hg": {"memory_turn": "Maine 10 January ko 'Effective Communication in the Workplace' pe ek workshop attend ki, aur isse meri active listening aur conflict resolution skills mein bahut help mili. Office mein inki practice kaise continue karoon?", "assistant_reply": "Jawab dene se pehle colleague ki baat ko short mein repeat kijiye, har meeting mein kam se kam ek clarifying question poochhiye, aur jab disagreement shuru ho toh solutions pe baat karne se pehle common goal bataiye.", "followup_turn": "Agar koi colleague meeting mein baar-baar mujhe beech mein toke toh kya karoon?", "followup_reply": "Calm rahiye aur kuch aisa kahiye: mujhe yeh point poora karne dijiye, phir main aapki baat zaroor sunna chahungi. Agar yeh continue ho, toh team ke saamne nahi balki akele mein unse baat kijiye.", "q1": "Main 'Effective Communication in the Workplace' wali workshop mein kab gayi thi?", "q2": "Jis communication workshop mein main gayi thi, woh kis date ko thi?"}}, "gold": {"answer_en": "January 10", "date": {"day": 10, "month": 1}, "slots": [{"name": "date", "aliases": ["January 10", "January 10th", "10 January", "10th January", "10th of January", "Jan 10", "10 Jan", "10 जनवरी"]}]}, "source": {"dataset": "LongMemEval", "question_id": "0bb5a684", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 4232278006, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E004", "cand_id": "EC012", "memory_type": "event", "user_gender": "male", "has_distractor": true, "text": {"en": {"memory_turn": "I attended the Holi celebration at my local temple on February 26th and had a blast throwing colours and learning about its significance. Can you tell me more about the story behind Holi?", "assistant_reply": "Holi is linked to the legend of Prahlad and Holika, which celebrates devotion and the victory of good over evil. It also marks the arrival of spring, which is why colours are such a big part of it.", "followup_turn": "How do I get the colour stains off my skin?", "followup_reply": "Wash with lukewarm water and a mild soap, then gently rub in some coconut oil or a besan and curd paste. Avoid scrubbing hard; the remaining colour fades in a day or two.", "q1": "When did I attend the Holi celebration at my temple?", "q2": "On what date did I go to the Holi celebration at the local temple?", "distractor_sentence": "My cousin went to a Holi party at her college on March 1st."}, "hi": {"memory_turn": "मैं 26 फ़रवरी को अपने यहाँ के मंदिर में होली के उत्सव में गया था, रंग खेलने और इसका महत्व जानने में बहुत मज़ा आया। होली के पीछे की कहानी के बारे में और बताएँगे?", "assistant_reply": "होली प्रह्लाद और होलिका की कथा से जुड़ी है, जो भक्ति और बुराई पर अच्छाई की जीत का उत्सव है। यह वसंत के आने का भी प्रतीक है, इसीलिए इसमें रंगों का इतना महत्व है।", "followup_turn": "त्वचा से रंग के दाग़ कैसे छुड़ाऊँ?", "followup_reply": "गुनगुने पानी और हल्के साबुन से धोइए, फिर धीरे-धीरे नारियल तेल या बेसन और दही का लेप लगाइए। ज़ोर से मत रगड़िए; बचा हुआ रंग एक-दो दिन में हल्का हो जाएगा।", "q1": "मैं अपने यहाँ के मंदिर में होली के उत्सव में कब गया था?", "q2": "मंदिर में होली के कार्यक्रम में मैं किस तारीख़ को गया था?", "distractor_sentence": "मेरी कज़िन 1 मार्च को अपने कॉलेज की होली पार्टी में गई थी।"}, "hg": {"memory_turn": "Main 26 February ko apne local temple mein Holi celebration mein gaya tha, rang khelne aur iski significance jaanne mein bahut maza aaya. Holi ke peeche ki kahani ke baare mein aur batayenge?", "assistant_reply": "Holi Prahlad aur Holika ki katha se judi hai, jo bhakti aur buraai pe acchai ki jeet ka celebration hai. Yeh spring ke aane ka bhi symbol hai, isiliye ismein rangon ka itna importance hai.", "followup_turn": "Skin se rang ke daag kaise chhudaun?", "followup_reply": "Lukewarm paani aur mild soap se dhoiye, phir dheere-dheere coconut oil ya besan aur dahi ka lep lagaiye. Zor se mat ragdiye; bacha hua rang ek-do din mein halka ho jayega.", "q1": "Main apne local temple mein Holi celebration mein kab gaya tha?", "q2": "Temple mein Holi ke function mein main kis date ko gaya tha?", "distractor_sentence": "Meri cousin 1 March ko apne college ki Holi party mein gayi thi."}}, "gold": {"answer_en": "February 26", "date": {"day": 26, "month": 2}, "slots": [{"name": "date", "aliases": ["February 26", "February 26th", "26 February", "26th February", "26th of February", "Feb 26", "26 Feb", "26 फ़रवरी", "26 फरवरी"]}]}, "distractor": {"date": {"day": 1, "month": 3}, "value_aliases": ["March 1", "March 1st", "1 March", "1st March", "1st of March", "Mar 1", "1 Mar", "1 मार्च"]}, "source": {"dataset": "LongMemEval", "question_id": "2a1811e2", "lme_type": "temporal-reasoning"}, "topic": "other", "is_answerable": true, "evidence_position": 2, "filler_seed": 450491531, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E005", "cand_id": "EC017", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "I ran 5 kilometres in the Run for the Cure event on October 15th and raised money for breast cancer research. It was an amazing experience! How can I build up to a 10K?", "assistant_reply": "Increase your weekly distance by about ten percent at a time, add one longer slow run each week, and include a rest day after hard sessions. Most runners can go from 5K to 10K in eight to ten weeks.", "followup_turn": "Should I run every day while training?", "followup_reply": "Not necessarily. Three or four runs a week with cross-training like cycling or yoga on other days lowers injury risk and still builds endurance.", "q1": "When did I run in the Run for the Cure event?", "q2": "On what date did I do the Run for the Cure 5K?"}, "hi": {"memory_turn": "मैंने 15 अक्टूबर को 'रन फ़ॉर द क्योर' इवेंट में 5 किलोमीटर दौड़ लगाई और ब्रेस्ट कैंसर रिसर्च के लिए पैसे जुटाए। बहुत शानदार अनुभव था! 10 किलोमीटर तक कैसे पहुँचूँ?", "assistant_reply": "हर हफ़्ते की दूरी एक बार में लगभग दस प्रतिशत बढ़ाइए, हफ़्ते में एक लंबी धीमी दौड़ जोड़िए, और कठिन अभ्यास के बाद एक दिन आराम कीजिए। ज़्यादातर लोग आठ से दस हफ़्तों में 5K से 10K तक पहुँच जाते हैं।", "followup_turn": "क्या ट्रेनिंग के दौरान रोज़ दौड़ना चाहिए?", "followup_reply": "ज़रूरी नहीं। हफ़्ते में तीन-चार बार दौड़ें और बाक़ी दिनों में साइकिलिंग या योग जैसी क्रॉस-ट्रेनिंग करें, इससे चोट का ख़तरा कम होता है और स्टैमिना भी बढ़ता है।", "q1": "मैंने 'रन फ़ॉर द क्योर' इवेंट में कब दौड़ लगाई थी?", "q2": "'रन फ़ॉर द क्योर' की 5K दौड़ मैंने किस तारीख़ को की थी?"}, "hg": {"memory_turn": "Maine 15 October ko 'Run for the Cure' event mein 5 kilometre run kiya aur breast cancer research ke liye paise collect kiye. Bahut amazing experience tha! 10K tak kaise pahunchoon?", "assistant_reply": "Har hafte ki distance ek baar mein lagbhag das percent badhaiye, hafte mein ek long slow run add kijiye, aur hard sessions ke baad ek din rest kijiye. Zyadatar runners aath se das hafton mein 5K se 10K tak pahunch jaate hain.", "followup_turn": "Kya training ke time roz run karna chahiye?", "followup_reply": "Zaroori nahi. Hafte mein teen-chaar runs aur baaki dino mein cycling ya yoga jaisi cross-training karein, isse injury ka risk kam hota hai aur stamina bhi badhta hai.", "q1": "Maine 'Run for the Cure' event mein kab run kiya tha?", "q2": "'Run for the Cure' ka 5K maine kis date ko kiya tha?"}}, "gold": {"answer_en": "October 15", "date": {"day": 15, "month": 10}, "slots": [{"name": "date", "aliases": ["October 15", "October 15th", "15 October", "15th October", "15th of October", "Oct 15", "15 Oct", "15 अक्टूबर", "15 अक्तूबर"]}]}, "source": {"dataset": "LongMemEval", "question_id": "a3838d2b", "lme_type": "temporal-reasoning"}, "topic": "fitness/health", "is_answerable": true, "evidence_position": 5, "filler_seed": 1940062016, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E006", "cand_id": "EC030", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I had a great time celebrating my best friend's 30th birthday party on the 22nd of April. She loved the personalized photo album I got her. Now I want to make a scrapbook for my parents. Any ideas?", "assistant_reply": "Organise it by decades or by family trips, add handwritten captions, and leave a few blank pages at the end so your parents can add their own memories later.", "followup_turn": "What paper should I use so the photos don't fade?", "followup_reply": "Use acid-free, lignin-free paper and photo-safe glue or corners. Keep the scrapbook away from direct sunlight to protect the colours.", "q1": "When was my best friend's 30th birthday party?", "q2": "On what date did I celebrate my best friend turning 30?"}, "hi": {"memory_turn": "22 अप्रैल को मैंने अपनी सबसे अच्छी दोस्त का 30वाँ जन्मदिन मनाया, बहुत मज़ा आया। उसे मेरा दिया हुआ पर्सनलाइज़्ड फ़ोटो एल्बम बहुत पसंद आया। अब मैं अपने मम्मी-पापा के लिए एक स्क्रैपबुक बनाना चाहता हूँ। कोई आइडिया?", "assistant_reply": "इसे दशकों या परिवार की यात्राओं के हिसाब से सजाइए, हाथ से लिखे कैप्शन जोड़िए, और आख़िर में कुछ ख़ाली पन्ने छोड़िए ताकि मम्मी-पापा बाद में अपनी यादें जोड़ सकें।", "followup_turn": "किस काग़ज़ का इस्तेमाल करूँ ताकि फ़ोटो फीकी न पड़ें?", "followup_reply": "एसिड-फ़्री और लिग्निन-फ़्री काग़ज़ और फ़ोटो के लिए सुरक्षित गोंद या कॉर्नर इस्तेमाल कीजिए। रंग बचाने के लिए स्क्रैपबुक को सीधी धूप से दूर रखिए।", "q1": "मेरी सबसे अच्छी दोस्त की 30वें जन्मदिन की पार्टी कब थी?", "q2": "मैंने अपनी बेस्ट फ़्रेंड का 30वाँ जन्मदिन किस तारीख़ को मनाया था?"}, "hg": {"memory_turn": "22 April ko maine apni best friend ka 30th birthday celebrate kiya, bahut maza aaya. Use mera diya hua personalized photo album bahut pasand aaya. Ab main apne mummy-papa ke liye ek scrapbook banana chahta hoon. Koi ideas?", "assistant_reply": "Ise decades ya family trips ke hisaab se arrange kijiye, haath se likhe captions add kijiye, aur end mein kuch blank pages chhodiye taaki mummy-papa baad mein apni yaadein add kar sakein.", "followup_turn": "Kaunsa paper use karoon taaki photos fade na hon?", "followup_reply": "Acid-free aur lignin-free paper aur photo-safe glue ya corners use kijiye. Colours bachane ke liye scrapbook ko direct sunlight se door rakhiye.", "q1": "Meri best friend ki 30th birthday party kab thi?", "q2": "Maine apni best friend ka 30th birthday kis date ko celebrate kiya tha?"}}, "gold": {"answer_en": "April 22", "date": {"day": 22, "month": 4}, "slots": [{"name": "date", "aliases": ["April 22", "April 22nd", "22 April", "22nd April", "22nd of April", "Apr 22", "22 Apr", "22 अप्रैल"]}]}, "source": {"dataset": "LongMemEval", "question_id": "a3045048", "lme_type": "temporal-reasoning"}, "topic": "media/books/games", "is_answerable": true, "evidence_position": 9, "filler_seed": 1586390893, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E007", "cand_id": "EC039", "memory_type": "event", "user_gender": "female", "has_distractor": false, "text": {"en": {"memory_turn": "My bike's gears were getting stuck, so I took it in for a tune-up on April 20th. Since then it rides much better. How often should I get a tune-up?", "assistant_reply": "For regular riders, a basic tune-up every six months is a good rule, with a full service once a year. Clean and lube the chain yourself every few weeks in between.", "followup_turn": "What's the easiest way to clean the chain at home?", "followup_reply": "Wipe it with a rag and degreaser while turning the pedals backwards, let it dry, then put a drop of chain lube on each link and wipe off the excess.", "q1": "When did I take my bike in for a tune-up?", "q2": "On what date did my bike get its tune-up?"}, "hi": {"memory_turn": "मेरी साइकिल के गियर अटक रहे थे, इसलिए मैं 20 अप्रैल को उसे ट्यून-अप के लिए ले गई थी। तब से वह बहुत अच्छी चल रही है। ट्यून-अप कितने समय बाद करवाना चाहिए?", "assistant_reply": "जो लोग नियमित रूप से चलाते हैं, उनके लिए हर छह महीने में बेसिक ट्यून-अप और साल में एक बार पूरी सर्विस ठीक रहती है। बीच-बीच में हर कुछ हफ़्तों में चेन ख़ुद साफ़ करके तेल डालिए।", "followup_turn": "घर पर चेन साफ़ करने का सबसे आसान तरीक़ा क्या है?", "followup_reply": "पैडल पीछे की ओर घुमाते हुए कपड़े और डीग्रीज़र से चेन पोंछिए, सूखने दीजिए, फिर हर कड़ी पर चेन ल्यूब की एक बूँद डालकर अतिरिक्त पोंछ दीजिए।", "q1": "मैं अपनी साइकिल ट्यून-अप के लिए कब ले गई थी?", "q2": "मेरी साइकिल का ट्यून-अप किस तारीख़ को हुआ था?"}, "hg": {"memory_turn": "Meri bike ke gears atak rahe the, isliye main 20 April ko use tune-up ke liye le gayi thi. Tab se woh bahut acchi chal rahi hai. Tune-up kitne time baad karwana chahiye?", "assistant_reply": "Regular riders ke liye har chhe mahine mein basic tune-up aur saal mein ek baar full service theek rehti hai. Beech-beech mein har kuch hafton mein chain khud saaf karke lube daaliye.", "followup_turn": "Ghar pe chain saaf karne ka sabse easy tareeka kya hai?", "followup_reply": "Pedals ko peeche ki taraf ghumaate hue kapde aur degreaser se chain ponchhiye, sookhne dijiye, phir har link pe chain lube ki ek boond daal kar extra ponchh dijiye.", "q1": "Main apni bike tune-up ke liye kab le gayi thi?", "q2": "Meri bike ka tune-up kis date ko hua tha?"}}, "gold": {"answer_en": "April 20", "date": {"day": 20, "month": 4}, "slots": [{"name": "date", "aliases": ["April 20", "April 20th", "20 April", "20th April", "20th of April", "Apr 20", "20 Apr", "20 अप्रैल"]}]}, "source": {"dataset": "LongMemEval", "question_id": "gpt4_d84a3211", "lme_type": "multi-session"}, "topic": "vehicles", "is_answerable": true, "evidence_position": 2, "filler_seed": 4104915379, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E008", "cand_id": "EC045", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I went on a guided tour at the History Museum on February 24th, and it really sparked my interest in ancient history and art. Can you suggest some beginner-friendly books on ancient civilisations?", "assistant_reply": "Good starting points are 'A Little History of the World' by E. H. Gombrich, 'Sapiens' by Yuval Noah Harari for a broad overview, and 'The Story of Art' by Gombrich for art history.", "followup_turn": "Which one should I read first?", "followup_reply": "Start with 'A Little History of the World'. It is short, easy to read, and gives you a timeline that makes the other two books easier to follow.", "q1": "When did I go on the guided tour at the History Museum?", "q2": "On what date did I visit the History Museum for a guided tour?"}, "hi": {"memory_turn": "मैं 24 फ़रवरी को हिस्ट्री म्यूज़ियम में एक गाइडेड टूर पर गया था, और इससे प्राचीन इतिहास और कला में मेरी रुचि जाग गई। प्राचीन सभ्यताओं पर शुरुआती लोगों के लिए कुछ किताबें सुझाएँगे?", "assistant_reply": "शुरुआत के लिए अच्छी किताबें हैं: ई. एच. गोम्ब्रिच की 'A Little History of the World', व्यापक नज़रिए के लिए युवाल नोआ हरारी की 'Sapiens', और कला इतिहास के लिए गोम्ब्रिच की 'The Story of Art'।", "followup_turn": "इनमें से पहले कौन-सी पढ़ूँ?", "followup_reply": "'A Little History of the World' से शुरू कीजिए। यह छोटी और आसान है, और इससे एक टाइमलाइन बन जाती है जिससे बाक़ी दोनों किताबें समझना आसान हो जाता है।", "q1": "मैं हिस्ट्री म्यूज़ियम में गाइडेड टूर पर कब गया था?", "q2": "मैं गाइडेड टूर के लिए हिस्ट्री म्यूज़ियम किस तारीख़ को गया था?"}, "hg": {"memory_turn": "Main 24 February ko History Museum mein ek guided tour pe gaya tha, aur isse ancient history aur art mein meri interest jaag gayi. Ancient civilisations pe beginners ke liye kuch books suggest karenge?", "assistant_reply": "Start ke liye acchi books hain: E. H. Gombrich ki 'A Little History of the World', broad overview ke liye Yuval Noah Harari ki 'Sapiens', aur art history ke liye Gombrich ki 'The Story of Art'.", "followup_turn": "Inmein se pehle kaunsi padhoon?", "followup_reply": "'A Little History of the World' se start kijiye. Yeh chhoti aur easy hai, aur isse ek timeline ban jaati hai jisse baaki dono books samajhna easy ho jaata hai.", "q1": "Main History Museum mein guided tour pe kab gaya tha?", "q2": "Main guided tour ke liye History Museum kis date ko gaya tha?"}}, "gold": {"answer_en": "February 24", "date": {"day": 24, "month": 2}, "slots": [{"name": "date", "aliases": ["February 24", "February 24th", "24 February", "24th February", "24th of February", "Feb 24", "24 Feb", "24 फ़रवरी", "24 फरवरी"]}]}, "source": {"dataset": "LongMemEval", "question_id": "2ce6a0f2", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 5, "filler_seed": 2034785533, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E009", "cand_id": "EC049", "memory_type": "event", "user_gender": "female", "has_distractor": true, "text": {"en": {"memory_turn": "I just turned 32 on February 12th, so I'm feeling more motivated to take care of myself. What are some simple health habits I could start this year?", "assistant_reply": "Start small: a 20-minute walk most days, a regular sleep time, more vegetables at lunch, and a yearly health check-up. Small habits you can keep beat big plans you drop.", "followup_turn": "What tests should a yearly check-up include at my age?", "followup_reply": "Commonly a blood count, blood sugar, cholesterol, thyroid and vitamin D, plus blood pressure. Your doctor may add others based on your family history.", "q1": "When did I turn 32?", "q2": "On what date is my birthday?", "distractor_sentence": "My brother turns 29 on August 8th, so we're already planning his party."}, "hi": {"memory_turn": "12 फ़रवरी को मैं 32 साल की हो गई, इसलिए अब अपनी सेहत का ध्यान रखने का मन ज़्यादा कर रहा है। इस साल कौन-सी आसान हेल्दी आदतें शुरू कर सकती हूँ?", "assistant_reply": "छोटे से शुरू कीजिए: ज़्यादातर दिनों में 20 मिनट की सैर, सोने का एक तय समय, दोपहर के खाने में ज़्यादा सब्ज़ियाँ, और साल में एक बार हेल्थ चेक-अप। जो छोटी आदतें टिक सकें, वे बड़े अधूरे प्लान से बेहतर हैं।", "followup_turn": "मेरी उम्र में सालाना चेक-अप में कौन-से टेस्ट होने चाहिए?", "followup_reply": "आमतौर पर ब्लड काउंट, ब्लड शुगर, कोलेस्ट्रॉल, थायरॉइड और विटामिन D, साथ में ब्लड प्रेशर। आपके परिवार के इतिहास के हिसाब से डॉक्टर कुछ और टेस्ट जोड़ सकते हैं।", "q1": "मैं 32 साल की कब हुई?", "q2": "मेरा जन्मदिन किस तारीख़ को है?", "distractor_sentence": "मेरा भाई 8 अगस्त को 29 साल का होगा, इसलिए हम अभी से उसकी पार्टी की तैयारी कर रहे हैं।"}, "hg": {"memory_turn": "12 February ko main 32 saal ki ho gayi, isliye ab apni health ka dhyan rakhne ka mann zyada kar raha hai. Is saal kaunsi easy healthy habits shuru kar sakti hoon?", "assistant_reply": "Chhote se shuru kijiye: zyadatar dino mein 20 minute ki walk, sone ka ek fixed time, lunch mein zyada sabziyan, aur saal mein ek baar health check-up. Jo chhoti habits tik sakein, woh bade adhoore plans se better hain.", "followup_turn": "Meri age mein yearly check-up mein kaunse tests hone chahiye?", "followup_reply": "Usually blood count, blood sugar, cholesterol, thyroid aur vitamin D, saath mein blood pressure. Aapki family history ke hisaab se doctor kuch aur tests add kar sakte hain.", "q1": "Main 32 saal ki kab hui?", "q2": "Mera birthday kis date ko hai?", "distractor_sentence": "Mera bhai 8 August ko 29 saal ka hoga, isliye hum abhi se uski party ki planning kar rahe hain."}}, "gold": {"answer_en": "February 12", "date": {"day": 12, "month": 2}, "slots": [{"name": "date", "aliases": ["February 12", "February 12th", "12 February", "12th February", "12th of February", "Feb 12", "12 Feb", "12 फ़रवरी", "12 फरवरी"]}]}, "distractor": {"date": {"day": 8, "month": 8}, "value_aliases": ["August 8", "August 8th", "8 August", "8th August", "8th of August", "Aug 8", "8 Aug", "8 अगस्त"]}, "source": {"dataset": "LongMemEval", "question_id": "gpt4_d12ceb0e", "lme_type": "multi-session"}, "topic": "other", "is_answerable": true, "evidence_position": 9, "filler_seed": 3481914820, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E010", "cand_id": "EC176", "memory_type": "event", "user_gender": "male", "has_distractor": false, "text": {"en": {"memory_turn": "I completed my first full marathon on April 10th and I'm feeling a bit sore. What should I do to recover properly?", "assistant_reply": "Congratulations! Rest for a few days, walk gently to keep blood flowing, eat enough protein and carbohydrates, and drink plenty of fluids. Wait about two weeks before any hard running.", "followup_turn": "Is a massage a good idea right now?", "followup_reply": "A light massage after two or three days can ease tightness. Avoid deep tissue work in the first days, when your muscles are still repairing.", "q1": "When did I complete my first full marathon?", "q2": "On what date did I run my first marathon?"}, "hi": {"memory_turn": "मैंने 10 अप्रैल को अपनी पहली फ़ुल मैराथन पूरी की और अब शरीर में थोड़ा दर्द है। ठीक से रिकवर होने के लिए क्या करूँ?", "assistant_reply": "बधाई हो! कुछ दिन आराम कीजिए, ख़ून का संचार बनाए रखने के लिए हल्का टहलिए, पर्याप्त प्रोटीन और कार्बोहाइड्रेट खाइए, और ख़ूब पानी पीजिए। किसी भी तेज़ दौड़ से पहले लगभग दो हफ़्ते रुकिए।", "followup_turn": "क्या अभी मसाज करवाना ठीक रहेगा?", "followup_reply": "दो-तीन दिन बाद हल्की मसाज से जकड़न कम हो सकती है। शुरुआती दिनों में गहरी मसाज से बचिए, जब मांसपेशियाँ अभी ठीक हो रही होती हैं।", "q1": "मैंने अपनी पहली फ़ुल मैराथन कब पूरी की थी?", "q2": "मैंने अपनी पहली मैराथन किस तारीख़ को दौड़ी थी?"}, "hg": {"memory_turn": "Maine 10 April ko apni pehli full marathon complete ki aur ab body mein thoda dard hai. Theek se recover hone ke liye kya karoon?", "assistant_reply": "Congratulations! Kuch din rest kijiye, blood flow bana rehne ke liye halka tehliye, kaafi protein aur carbs khaiye, aur khoob paani peejiye. Kisi bhi hard run se pehle lagbhag do hafte rukiye.", "followup_turn": "Kya abhi massage karwana theek rahega?", "followup_reply": "Do-teen din baad light massage se stiffness kam ho sakti hai. Shuru ke dino mein deep tissue massage se bachiye, jab muscles abhi repair ho rahi hoti hain.", "q1": "Maine apni pehli full marathon kab complete ki thi?", "q2": "Maine apni pehli marathon kis date ko daudi thi?"}}, "gold": {"answer_en": "April 10", "date": {"day": 10, "month": 4}, "slots": [{"name": "date", "aliases": ["April 10", "April 10th", "10 April", "10th April", "10th of April", "Apr 10", "10 Apr", "10 अप्रैल"]}]}, "source": {"dataset": "LongMemEval", "question_id": "21d02d0d", "lme_type": "multi-session"}, "topic": "fitness/health", "is_answerable": true, "evidence_position": 2, "filler_seed": 471514393, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "F001-ABS", "base_item": "F001", "memory_type": "fact", "is_answerable": false, "text": {"en": {"q1": "Which audiobook app do I use on my commute?", "q2": "What's the name of the app I listen to audiobooks on?"}, "hi": {"q1": "मैं सफ़र में ऑडियोबुक्स सुनने के लिए कौन-सा ऐप इस्तेमाल करती हूँ?", "q2": "जिस ऐप पर मैं ऑडियोबुक्स सुनती हूँ, उसका नाम क्या है?"}, "hg": {"q1": "Main commute mein audiobooks sunne ke liye kaunsa app use karti hoon?", "q2": "Jis app pe main audiobooks sunti hoon, uska naam kya hai?"}}, "gold": {"answer_en": "NO_ANSWER (never mentioned; Kindle is mentioned only for reading a book)"}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "P003-ABS", "base_item": "P003", "memory_type": "preference", "is_answerable": false, "text": {"en": {"q1": "Which laptop do I edit my videos on?", "q2": "What laptop do I have for video editing?"}, "hi": {"q1": "मैं अपने वीडियो किस लैपटॉप पर एडिट करती हूँ?", "q2": "वीडियो एडिटिंग के लिए मेरे पास कौन-सा लैपटॉप है?"}, "hg": {"q1": "Main apne videos kis laptop pe edit karti hoon?", "q2": "Video editing ke liye mere paas kaunsa laptop hai?"}}, "gold": {"answer_en": "NO_ANSWER (laptop never mentioned)"}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}
{"item_id": "E001-ABS", "base_item": "E001", "memory_type": "event", "is_answerable": false, "text": {"en": {"q1": "When did I get my car detailed?", "q2": "On what date was my car's detailing done?"}, "hi": {"q1": "मैंने अपनी कार की डिटेलिंग कब करवाई थी?", "q2": "मेरी कार की डिटेलिंग किस तारीख़ को हुई थी?"}, "hg": {"q1": "Maine apni car ki detailing kab karwayi thi?", "q2": "Meri car ki detailing kis date ko hui thi?"}}, "gold": {"answer_en": "NO_ANSWER (detailing was only planned, never done; the service date is a trap)"}, "drafted_by": "Claude Opus 5.5 (claude-opus-5-5), 3 Oct 2026"}


In [ ]:
%%writefile /kaggle/working/clm/data/haystacks/filler_pool_en.jsonl
{"filler_id": "FL01", "session_id": "ultrachat_201585", "turns": [{"role": "user", "content": "In what ways did the Beatles' fame and exposure affect Lennon's songwriting?"}, {"role": "assistant", "content": "As an AI language model, I don't have personal beliefs, but I can give you a general answer. The Beatles' fame and exposure had a profound impact on Lennon's songwriting. With the band's massive success, Lennon was given a platform to share his music with the world, and he used it to write some of the most influential songs of all time."}, {"role": "user", "content": "Can you give me some examples of specific songs that were influenced by Lennon's fame and exposure?"}, {"role": "assistant", "content": "Sure, here are some examples of Lennon's songs that were influenced by his fame and exposure: 1. \"Help!\" - This song was written during a time when Lennon was feeling overwhelmed by the band's success and the constant demands placed on him as a public figure. 2. \"Strawberry Fields Forever\" - This song was inspired by Lennon's memories of growing up near Strawberry Field, a Salvation Army children's home."}]}
{"filler_id": "FL02", "session_id": "5fb7fd18_1", "turns": [{"role": "user", "content": "I'm looking for some new baking inspiration. I've been doing a lot of sweet treats lately, but I want to try something savory. Do you have any recommendations for a good savory bread recipe? By the way, I enjoyed my sourdough bread with a pat of butter the last time I made it, and I'm hoping to recreate that flavor."}, {"role": "assistant", "content": "A savory bread enthusiast in the making! I'd be delighted to help you explore the world of savory breads. Since you mentioned enjoying sourdough with butter, I'll suggest some recipes that share similar characteristics or flavor profiles. Here are a few ideas to get you started: 1. Rosemary and Olive Oil Focaccia: This Italian bread is perfect for sopping up olive oil, much like your sourdough with butter."}, {"role": "user", "content": "These recipes sound amazing, especially the Rosemary and Olive Oil Focaccia. I've never worked with rosemary in bread before, so I'm excited to try it out. Can you give me some tips on how to handle fresh rosemary and how much to use in the recipe?"}, {"role": "assistant", "content": "Working with fresh rosemary can be a bit tricky, but with some tips, you'll be a pro in no time! Handling Fresh Rosemary: 1. Choose fresh, fragrant sprigs: Select rosemary sprigs with a vibrant green color and a potent, piney aroma. Avoid wilted or brownish sprigs, as they can be bitter. 2. Store rosemary properly: Keep fresh rosemary in a plastic bag or wrap it in plastic wrap in the refrigerator to maintain its flavor and aroma."}]}
{"filler_id": "FL03", "session_id": "afbd7193_2", "turns": [{"role": "user", "content": "I'm looking for some tips on organizing my bookshelf. I just did a deep clean of my living room last Sunday and realized I need some ideas on how to maximize my bookshelf space. By the way, I commit to a 30-minute deep clean session every Sunday, so I want to make the most out of it."}, {"role": "assistant", "content": "Kudos to you for committing to a regular deep clean session! That's a great habit to maintain a tidy and organized living space. Now, let's dive into some tips to help you maximize your bookshelf space and make the most of your 30-minute Sunday deep clean session: 1. Purge and categorize: Start by removing all books from your shelf and sorting them into categories (e.g., fiction, non-fiction, biographies, self-help, etc.)."}, {"role": "user", "content": "I like the idea of categorizing and assigning a home for each category. But how do I decide what to keep and what to get rid of? I have a lot of books that I haven't read in years, but they hold sentimental value."}, {"role": "assistant", "content": "Deciding what to keep and what to let go of can be a challenging task, especially when sentimental value is involved. Here are some tips to help you make that decision: Sentimental value vs. practicality: 1. Ask yourself: When was the last time I looked at or read this book? Be honest. If it's been years, it's likely you won't miss it if you let it go. 2."}]}
{"filler_id": "FL04", "session_id": "ultrachat_391062", "turns": [{"role": "user", "content": "What are some common gaming terms and acronyms that I should be familiar with?"}, {"role": "assistant", "content": "Here are some common gaming terms and acronyms: 1. FPS - First-person shooter 2. MMO - Massively multiplayer online 3. PVP - Player versus player 4. NPC - Non-player character 5. RNG - Random number generator 6. MOBA - Multiplayer online battle arena 7. DLC - Downloadable content 8. LFG - Looking for group 9. AFK - Away from keyboard 10. DPS - Damage per second 11."}, {"role": "user", "content": "Dude, I'm so sick of the RNG in this game. I've been grinding for hours and still haven't gotten the drop I need. It's like the game is rigged against me."}, {"role": "assistant", "content": "I understand that the user is expressing frustration with a game due to rng (random number generator), meaning the outcome of certain events or drops is determined randomly rather than based on skill. the user is unhappy that after spending hours in the game and grinding, they still haven't obtained the desired drop which they believe to be due to poor luck or unfair odds."}]}
{"filler_id": "FL05", "session_id": "ultrachat_274962", "turns": [{"role": "user", "content": "How did Sulla's military victories and conquests shape his political ideology and influence his policies as a leader in Rome?"}, {"role": "assistant", "content": "Sulla's military victories and conquests played a significant role in shaping his political ideology and influencing his policies as a leader in Rome. As a military general, Sulla was known for his exceptional skills and tactical brilliance on the battlefield. Through his military successes, he gained immense popularity among the Roman people and established a power base that he would later use to seize control of the state."}, {"role": "user", "content": "It seems like Sulla was more interested in consolidating his own power than truly serving the Roman people. What impact did his policies have on the citizens of Rome?"}, {"role": "assistant", "content": "You are correct in stating that Sulla was more interested in consolidating his own power than serving the Roman people. His policies had a significant impact on the citizens of Rome that ultimately contributed to the decline of the Roman Republic. One of the most significant impacts of Sulla's policies was the centralization of power in the hands of a few individuals."}]}
{"filler_id": "FL06", "session_id": "f62c04c6", "turns": [{"role": "user", "content": "I'm thinking of creating a content calendar to plan out my social media posts for the next month. Can you help me with some ideas on what types of posts I should make and how often I should post?"}, {"role": "assistant", "content": "Creating a content calendar is a great way to plan and organize your social media content in advance. The type and frequency of posts will depend on your target audience, goals, and the platforms you're using, but here are some general ideas and guidelines to get you started: Post Types: 1. Educational/Informative: Share tips, tutorials, industry news, or how-to guides related to your niche. 2. Promotional: Share updates about your products, services, or events. 3."}, {"role": "user", "content": "That's a lot of great info! I think I'll start by creating a template for each platform and then filling it in with ideas. Can you give me some ideas for Instagram Reels? I recently posted one about my morning yoga routine and got 50 views and 10 likes, so I'm thinking of doing a series on yoga or meditation."}, {"role": "assistant", "content": "Instagram Reels are a fantastic way to engage with your audience and increase visibility. Congratulations on your first Reel's success! A yoga or meditation series is an excellent idea, especially since you've already seen interest in your morning yoga routine Reel. Here are some ideas to get you started: Yoga Series: 1. Morning Routine Variations: Share different morning yoga routines, each focusing on a specific area, like flexibility, strength, or relaxation. 2."}]}
{"filler_id": "FL07", "session_id": "8969b3ca_2", "turns": [{"role": "user", "content": "I'm looking to get an estimate of the total costs involved in buying a house. Can you help me with that?"}, {"role": "assistant", "content": "Estimating the total costs involved in buying a house can help you plan and budget for this significant investment. Here's a breakdown of the typical costs you can expect to incur: 1. Purchase Price: This is the biggest cost of buying a house, and it's the amount you'll pay to the seller for the property. 2. Down Payment: This is a percentage of the purchase price that you'll pay upfront, typically ranging from 3.5% to 20% of the purchase price."}, {"role": "user", "content": "I'm actually getting close to closing on a house, and I'd like to get a more accurate estimate of my closing costs. The sellers agreed to give me a credit of $2,000 towards closing costs, so I'd like to factor that in. Can you help me with that?"}, {"role": "assistant", "content": "Congratulations on getting close to closing on your new home! I'd be happy to help you estimate your closing costs and factor in the $2,000 credit from the sellers. To get an accurate estimate, I'll need some more information from you. Please provide the following details: 1. Purchase price: What is the final purchase price of the house? 2. Loan amount: How much are you borrowing to finance the purchase? 3. Loan type: What type of loan are you getting?"}]}
{"filler_id": "FL08", "session_id": "50d66391_7", "turns": [{"role": "user", "content": "I'm looking for some advice on bedroom lighting. I've been thinking about getting some floor lamps for my bedroom to create a relaxing reading nook."}, {"role": "assistant", "content": "What a lovely idea! Floor lamps can be a great addition to a bedroom, creating a cozy and inviting atmosphere perfect for a relaxing reading nook. Here are some tips to consider when choosing the right floor lamps for your bedroom: 1. Consider the purpose: Since you mentioned a reading nook, you'll want to prioritize task lighting. Look for lamps with adjustable arms or heads that can direct light exactly where you need it. 2."}, {"role": "user", "content": "I'm currently using a bedside lamp with a new bulb that I got from Home Depot for around $15, and I really like its warm tone. Do you think I should look for floor lamps with similar bulb options or color temperature?"}, {"role": "assistant", "content": "You've already found a bulb you like, which is a great starting point! Yes, it's a good idea to look for floor lamps that can accommodate similar bulb options or have a similar color temperature to the one you're currently using. Since you mentioned you like the warm tone of the bulb you got from Home Depot, I'm assuming it's likely in the range of 2700K-3000K, which is a warm white or soft white color temperature."}]}
{"filler_id": "FL09", "session_id": "ultrachat_436464", "turns": [{"role": "user", "content": "What are the key initiatives taken by World Health Organization (WHO) to improve public health across the globe?"}, {"role": "assistant", "content": "The World Health Organization (WHO) has undertaken numerous key initiatives to improve public health across the globe, including: 1. Universal Health Coverage: WHO aims to ensure that everyone can access the health services they need without facing financial hardship. This includes expanding access to essential health services and addressing health inequalities. 2. Global Vaccine Action Plan: WHO works to ensure that all people, regardless of where they live, have access to life-saving vaccines."}, {"role": "user", "content": "It's great to see all these initiatives by WHO. But do you think they are effective in improving public health?"}, {"role": "assistant", "content": "I do not have personal opinions. however, who's initiatives have shown significant progress in improving public health worldwide. for example, the global vaccine action plan has contributed to increasing vaccination coverage in many countries, and the tobacco free initiative has been successful in reducing the demand for tobacco products. similarly, who's efforts in disease control, emergency response, and mental health promotion have also led to positive outcomes."}]}
{"filler_id": "FL10", "session_id": "429be85c_4", "turns": [{"role": "user", "content": "I'm looking to try out some new breakfast treats to pair with my coffee on weekends. I like to indulge in a bigger coffee break with some breakfast treats on weekends, and I'm getting a bit tired of my usual cinnamon rolls. Do you have any recipe suggestions or recommendations?"}, {"role": "assistant", "content": "I'm happy to help! Indulging in a delicious breakfast treat with a warm cup of coffee on weekends is one of life's greatest pleasures. I've got some scrumptious recipe suggestions to help you mix things up and find a new favorite breakfast companion. Here are a few ideas to get you started: 1. Cranberry Orange Scones: Flaky, buttery scones infused with dried cranberries and orange zest are a perfect match for a rich cup of coffee. 2."}, {"role": "user", "content": "What's the difference between using unsalted butter and salted butter in these recipes? Does it really make a difference in the final taste and texture?"}, {"role": "assistant", "content": "Using unsalted butter versus salted butter can indeed make a difference in the final taste and texture of your baked goods."}]}
{"filler_id": "FL11", "session_id": "33994682_3", "turns": [{"role": "user", "content": "I'm considering freezing my eggs soon and I've been doing some research, but I'm not sure about the process and costs involved. Can you walk me through the general procedure and give me an idea of how much it typically costs? By the way, my sister, who's only three years older than me, is 35 now, and it's crazy to think about how fast time has flown by."}, {"role": "assistant", "content": "I'd be happy to help you understand the process and costs involved in egg freezing! The Process: Egg freezing, also known as oocyte cryopreservation, is a multi-step process that typically takes around 2-3 weeks. Here's an overview: 1. Initial Consultation: You'll meet with a fertility specialist (reproductive endocrinologist) to discuss your medical history, fertility goals, and any concerns you may have. 2."}, {"role": "user", "content": "I'm also curious to know more about the success rates of egg freezing. What are the chances of a frozen egg resulting in a healthy pregnancy and birth, and does my age play a significant role in this?"}, {"role": "assistant", "content": "Success Rates of Egg Freezing: The success rates of egg freezing are improving with advancements in technology and vitrification methods. However, it's essential to understand that egg freezing is not a guarantee of a successful pregnancy or birth. The chances of a frozen egg resulting in a healthy pregnancy and birth depend on several factors, including: 1."}]}
{"filler_id": "FL12", "session_id": "f75238a5_2", "turns": [{"role": "user", "content": "I've been thinking about exploring AI tools for my project, and I was wondering if you could recommend some resources for learning about the latest developments in AI. By the way, I recently had a great conversation with someone about this topic - we discussed the latest developments in AI for about 10 minutes, and it really sparked my interest."}, {"role": "assistant", "content": "I'm thrilled to hear that you're interested in exploring AI tools for your project! There are indeed many exciting developments in the field of AI, and staying up-to-date with the latest advancements can be incredibly valuable. As for resources, I'd be happy to recommend some top-notch ones to get you started: 1. The AI Alignment Podcast: A fantastic podcast that delves into the latest AI research, applications, and implications."}, {"role": "user", "content": "I'll definitely check out those resources, especially the podcast and the blog by Emerj. Can you tell me more about the current state of AI applications in industries like healthcare and finance?"}, {"role": "assistant", "content": "AI is transforming industries across the board, and healthcare and finance are no exceptions. Here's a brief overview of the current state of AI applications in these sectors: Healthcare: 1. Disease diagnosis: AI-powered systems can analyze medical images, such as X-rays and MRIs, to detect diseases like cancer, diabetes, and cardiovascular disease with high accuracy. 2. Personalized medicine: AI helps tailor treatment plans to individual patients based on their genetic profiles, medical histories, and lifestyle factors. 3."}]}
{"filler_id": "FL13", "session_id": "ultrachat_364297", "turns": [{"role": "user", "content": "Can you compare and contrast the breeding habits of ducks and geese?"}, {"role": "assistant", "content": "I do not have personal experiences, emotions, or beliefs. however, i can provide you the information you need. ducks and geese are both waterfowl and have some similarities in breeding habits, but there are also some differences. here are some comparisons and contrasts: similarities: - both ducks and geese form pair bonds during the breeding season. - both species typically mate for life or long-term, although some will switch partners."}, {"role": "user", "content": "Can you tell me more about the courtship displays of ducks? It sounds interesting."}, {"role": "assistant", "content": "Sure, I'd be happy to! Ducks are known for their elaborate courtship displays, which are basically a series of behaviors used to attract a mate. Here are some examples of courtship displays that you might see in ducks: - Head-throws: Some species of ducks will toss their heads back and forth and make various sounds to show off their physical prowess and vocal abilities."}]}
{"filler_id": "FL14", "session_id": "08ca1f31_2", "turns": [{"role": "user", "content": "I'm thinking of planning a longer bike trip on my Trek Emonda this weekend, maybe 80-100 miles. Can you help me with some route suggestions or tips for preparing my bike for a longer ride? By the way, I recently had a tune-up done on it about 3 weeks ago, and the bike shop replaced the chain and adjusted the derailleurs, which cost me $80."}, {"role": "assistant", "content": "A longer bike trip sounds like a great adventure! I'd be happy to help you with route suggestions and tips for preparing your bike. Route Suggestions: Before we dive into bike prep, let's talk routes. Since you didn't specify a location, I'll provide some general suggestions. You can use online tools like MapMyRide, Ride with GPS, or Strava to find routes in your area. 1."}, {"role": "user", "content": "I was thinking of doing a route with some hills, as I've been doing a lot of flat commutes on my Trek Emonda lately. Do you have any specific hill-climbing tips or recommendations for gearing and cadence?"}, {"role": "assistant", "content": "Hill-climbing is a great way to challenge yourself and improve your overall cycling fitness. Here are some hill-climbing tips, gearing recommendations, and cadence advice to help you tackle those inclines on your Trek Emonda: Hill-Climbing Tips: 1. Pace yourself: Start with a steady, moderate pace, and try to maintain it throughout the climb. Avoid surging or sprinting, as this can lead to exhaustion. 2."}]}
{"filler_id": "FL15", "session_id": "1c1f5ccc_6", "turns": [{"role": "user", "content": "I'm interested in learning more about the cultural significance of Indian clothing, particularly the scarf I recently bought at a festival. Can you tell me more about the history and symbolism behind traditional Indian scarves? By the way, I've been trying to learn more about my own cultural heritage, including the history and customs of India, and this seems like a great place to start."}, {"role": "assistant", "content": "What a wonderful topic! Indian clothing, particularly the scarf you're interested in, holds a rich cultural significance that dates back centuries. I'm more than happy to delve into the history and symbolism behind traditional Indian scarves, and I'm thrilled to hear that you're exploring your cultural heritage. In India, scarves are an integral part of traditional attire, and they're known by different names depending on the region. Some common types of Indian scarves include: 1."}, {"role": "user", "content": "I'd like to know more about the different types of fabrics used in traditional Indian scarves. Are there any specific fabrics that are considered more symbolic or prestigious than others?"}, {"role": "assistant", "content": "Fabrics play a vital role in traditional Indian scarves, and each type has its own unique characteristics, significance, and cultural associations. Let me introduce you to some of the most common and revered fabrics used in Indian scarves: 1. Silk: Silk is considered the most prestigious and luxurious fabric in Indian textiles. It's often associated with royalty, wealth, and grandeur. Silk scarves are highly prized for their softness, luster, and durability."}]}
{"filler_id": "FL16", "session_id": "0651f7bd_2", "turns": [{"role": "user", "content": "I'm looking for some tips on how to properly store and display my vintage camera collection. I've got a few rare ones and I want to make sure they're protected from dust and damage. By the way, I got a rare card of Cal Ripken Jr. from 1982 in the mail today, but that's a whole other collection of mine."}, {"role": "assistant", "content": "Congratulations on your rare Cal Ripken Jr. card! That's exciting! Now, let's focus on storing and displaying your vintage camera collection. Proper storage and display are crucial to preserving the condition and value of your cameras. Here are some tips to help you keep your vintage cameras protected from dust and damage: Storage: 1. Climate control: Store your cameras in a room with a consistent temperature between 60°F to 70°F (15°C to 21°C) and humidity below 60%."}, {"role": "user", "content": "Can you recommend some good resources for learning about camera restoration and maintenance? I've got a few cameras that need some TLC, and I'm not sure where to start."}, {"role": "assistant", "content": "Camera restoration and maintenance are essential skills for any camera enthusiast. Here are some excellent resources to help you learn and get started: Online Forums and Communities: 1. The Camera Repair Forum: A dedicated forum with a vast community of experts and enthusiasts sharing knowledge and experiences on camera repair and maintenance. 2. Photography Stack Exchange: A Q&A platform where you can ask questions and get answers from experienced photographers and camera enthusiasts. 3."}]}
{"filler_id": "FL17", "session_id": "01d4b26b_1", "turns": [{"role": "user", "content": "I'm looking for some gift ideas for a coworker's baby shower. I want something eco-friendly and cute. By the way, I just gave my sister a silver necklace with a small pendant for her birthday last month, and she loved it."}, {"role": "assistant", "content": "What a lovely gift for your sister! I'm happy to help you with some eco-friendly and adorable gift ideas for your coworker's baby shower. Here are some suggestions: 1. Organic Cotton Onesie and Baby Blanket Set: A soft, eco-friendly onesie and blanket set made from organic cotton. You can even opt for a set with a cute design or a personalized message. 2. Bamboo Baby Toys: Bamboo is a sustainable, eco-friendly material that's perfect for baby toys."}, {"role": "user", "content": "I like the organic cotton onesie and baby blanket set idea. Can you tell me more about the brands that offer customization options?"}, {"role": "assistant", "content": "Organic cotton onesie and baby blanket sets make for a thoughtful and eco-friendly gift. Several brands offer customization options for onesies and baby blankets. Here are a few popular ones: 1. Hanna Andersson: Known for their soft, organic cotton clothing, Hanna Andersson offers customization options for onesies, sleepers, and blankets. You can choose from various fonts, colors, and designs to create a personalized gift. 2."}]}
{"filler_id": "FL18", "session_id": "e2691068", "turns": [{"role": "user", "content": "I'm looking for some new breakfast ideas that can give me a energy boost. Do you have any suggestions?"}, {"role": "assistant", "content": "A energizing breakfast is just what you need to start your day off right! I've got some delicious and nutritious breakfast ideas that'll give you a boost of energy to tackle your day: 1. Avocado Toast with Poached Eggs: Whole grain toast topped with mashed avocado, a poached egg, and a sprinkle of red pepper flakes. The healthy fats from avocado and protein from the egg will keep you full and focused. 2."}, {"role": "user", "content": "I like the sound of the Avocado Toast with Poached Eggs and the Smoothie Bowl. Do you have any tips on how to prepare them quickly in the morning?"}, {"role": "assistant", "content": "Great choices! Both Avocado Toast with Poached Eggs and Smoothie Bowl are delicious and nutritious breakfast options that can be prepared quickly in the morning. Here are some tips to help you prepare them quickly: Avocado Toast with Poached Eggs: 1. Prep ahead: Mash the avocado and store it in an airtight container in the fridge overnight. This will save you about 5 minutes in the morning. 2."}]}
{"filler_id": "FL19", "session_id": "ultrachat_415674", "turns": [{"role": "user", "content": "How have advancements in neuroscience and psychology changed our understanding of the human mind and behavior?"}, {"role": "assistant", "content": "Advancements in neuroscience and psychology have revolutionized our understanding of the human mind and behavior in a number of ways: 1. Understanding of the Brain: Advancements in neuroimaging technology, such as functional magnetic resonance imaging (fMRI) and electroencephalogram (EEG), have allowed us to measure and better understand brain activity. This technology has allowed researchers to investigate how different parts of the brain are involved in various cognitive functions, such as perception, attention, emotion, decision-making, and social cognition. 2."}, {"role": "user", "content": "It's fascinating to see how developments in neuroscience and psychology have led to new insights on the human mind and behavior. What do you think will be the next breakthrough in these fields?"}, {"role": "assistant", "content": "I do not have personal opinions or the ability to predict future events. however, based on ongoing research and recent developments, there are several areas where we might expect to see significant breakthroughs in neuroscience and psychology in the near future. one of the areas of research that is receiving increasing attention is the study of the gut-brain axis."}]}
{"filler_id": "FL20", "session_id": "ultrachat_93661", "turns": [{"role": "user", "content": "What are some best practices for staying organized and managing time effectively in a fast-paced work environment?"}, {"role": "assistant", "content": "1. Use a planner or calendar: Write down all of your tasks and deadlines in one place to help you stay on top of your schedule and prevent things from slipping through the cracks. 2. Prioritize your tasks: Prioritize your work by importance and deadline. Start with the most critical work first, and then work your way down to less important tasks. 3."}, {"role": "user", "content": "I especially struggle with eliminating distractions, so I'll definitely work on that. Do you have any recommendations for specific productivity apps or tools?"}, {"role": "assistant", "content": "Yes, there are many productivity apps and tools available that can help you stay organized and manage your time effectively. Here are some recommendations: 1. Trello: A visual project management tool that allows you to create boards, cards, and lists to organize your tasks and workflows. 2. Focus@will: A music streaming service that plays background music designed to help you stay focused and productive. 3."}]}
{"filler_id": "FL21", "session_id": "b7c5c12f_2", "turns": [{"role": "user", "content": "I'm looking for some recommendations on sustainable living. I've been reading a lot about it lately, especially in my weekly magazine that I've been receiving every week. Anyway, can you give me some tips on how to reduce my carbon footprint in daily life?"}, {"role": "assistant", "content": "Sustainable living is such an important topic! I'm more than happy to help you with that. Here are some practical tips to reduce your carbon footprint in daily life: 1. Reduce Energy Consumption Turn off lights, electronics, and appliances when not in use to avoid standby power consumption. Switch to energy-efficient LED bulbs. Adjust your thermostat to use less energy for heating and cooling. 2. Conserve Water Take shorter showers (5-10 minutes) to save water and energy."}, {"role": "user", "content": "I'm actually quite interested in the sustainable fashion aspect, which I recently read about in my magazine. Can you give me some tips on how to make my wardrobe more eco-friendly?"}, {"role": "assistant", "content": "Sustainable fashion is a wonderful area to focus on! The fashion industry has a significant environmental impact, but by making a few conscious choices, you can reduce your fashion footprint. Here are some tips to help you create a more eco-friendly wardrobe: 1. Buy Second-Hand First Thrift stores, consignment shops, and online marketplaces like eBay, ThredUp, or Poshmark are great resources for second-hand shopping. You can find unique, high-quality pieces while reducing the demand for new, resource-intensive clothing. 2."}]}
{"filler_id": "FL22", "session_id": "df6970d8", "turns": [{"role": "user", "content": "I'm looking for recommendations on new board games to play with my friends and family. Any suggestions?"}, {"role": "assistant", "content": "New board games to play with friends and family? I've got a treasure trove of suggestions for you! Here are some modern classics, hidden gems, and popular games across various genres: Strategy and Competition 1. Gloomhaven: A cooperative campaign-based game with modular board and Legacy-style gameplay. 2. Scythe: A worker placement game set in an alternate history of the 1920s, with engine-building and area control. 3."}, {"role": "user", "content": "I've played Ticket to Ride and Settlers of Catan before, and they're great games. Have you got any recommendations for games that are similar to Scattergories?"}, {"role": "assistant", "content": "Scattergories is a fantastic game! If you enjoy the creative, word-based gameplay, here are some recommendations that might scratch that same itch: 1. Codenames: While not exactly like Scattergories, Codenames involves word-guessing and creative thinking. Players try to identify words based on one-word clues given by their teammate. 2. Quelf: A party game that challenges players to come up with unique words or phrases that fit certain criteria, often with hilarious results. 3."}]}
{"filler_id": "FL23", "session_id": "ultrachat_518652", "turns": [{"role": "user", "content": "What was the significance of the Trojan War in Greek mythology?"}, {"role": "assistant", "content": "The Trojan War was one of the most important events in Greek mythology. It was a war fought between the Greeks and the Trojans, and it lasted for ten years. The war was fought over Helen of Troy, who was the wife of Menelaus, the king of Sparta. Helen had been abducted by Paris, the prince of Troy, and taken to Troy. The war was significant because it was said to have been fought for honor and glory."}, {"role": "user", "content": "Which side ultimately won the Trojan War?"}, {"role": "assistant", "content": "The Greeks ultimately won the Trojan War by using the Trojan Horse, a wooden horse filled with Greek warriors, to invade and conquer the city of Troy. The Trojan Horse was left outside the gates of Troy as a supposed offering to the gods, and the Trojans brought it inside the city. That night, the Greek soldiers emerged from the horse and opened the gates of Troy, allowing the Greek army to enter and conquer the city."}]}
{"filler_id": "FL24", "session_id": "ultrachat_106352", "turns": [{"role": "user", "content": "What is the recommended amount of sleep per night for a healthy adult?"}, {"role": "assistant", "content": "The recommended amount of sleep per night for a healthy adult is 7-9 hours."}, {"role": "user", "content": "That sounds like a lot of time to spend sleeping. Can't I just get by with 5-6 hours?"}, {"role": "assistant", "content": "As an AI language model, I cannot give medical advice or suggest making changes to health routines. However, studies have shown that getting less than 7 hours of sleep consistently can lead to negative health effects, including decreased cognitive function, fatigue, and a weakened immune system. It is recommended that adults aim for at least 7 hours of sleep per night to maintain good health."}]}
{"filler_id": "FL25", "session_id": "fb793c87_2", "turns": [{"role": "user", "content": "I'm looking for some new shows to watch on HBO Max. I've been meaning to check out \"Game of Thrones\" for a while, and I finally started it about a month ago. I've finished the first four seasons so far, and I'm really enjoying it. Do you have any other recommendations similar to \"Game of Thrones\"?"}, {"role": "assistant", "content": "\"Game of Thrones\" is an epic show! You're halfway through, and I'm sure you're hooked. If you're looking for similar shows on HBO Max, I've got some fantastic recommendations for you: 1. Westworld: Another HBO epic, set in a futuristic theme park where guests can interact with lifelike robots. It's a complex, thought-provoking series with a similar blend of action, drama, and politics. 2. Vikings: A historical drama that follows the adventures of Ragnar Lothbrok, a legendary Viking warrior."}, {"role": "user", "content": "That's a great list! I'll definitely check some of them out. I've also been watching some documentaries on Hulu, particularly the ones on true crime and science. Do you have any recommendations for documentaries on HBO Max?"}, {"role": "assistant", "content": "HBO Max has an impressive collection of documentaries, and I've got some fantastic recommendations for you: True Crime: 1. I'll Be Gone in the Dark: A documentary series based on Michelle McNamara's book about the Golden State Killer. It's a gripping and thought-provoking exploration of the case and its impact on the victims and their families. 2."}]}
{"filler_id": "FL26", "session_id": "b0b73826", "turns": [{"role": "user", "content": "I'm trying to plan a get-together with my friends this weekend. Do you have any recommendations for a good brunch spot?"}, {"role": "assistant", "content": "I'd be happy to help! However, I'm a large language model, I don't have personal experiences or real-time information about specific locations. But I can provide you with some general tips and questions to help you find a great brunch spot for your gathering: 1. Consider your friends' preferences: Ask your friends about their dietary restrictions, favorite cuisines, and preferred ambiance. This will help you narrow down options. 2."}, {"role": "user", "content": "I was thinking of trying out a new place, but now I'm considering going back to a spot I've been to recently with my parents. It had a really great atmosphere and delicious food."}, {"role": "assistant", "content": "That's a great idea! Going back to a place you've been to before can be a great option, especially if you know and love the atmosphere and food. It's always nice to have a reliable spot that you can count on to deliver a great experience. Plus, since you've been there with your parents, you might have a special connection to the place, and it could be fun to share that experience with your friends."}]}
{"filler_id": "FL27", "session_id": "ultrachat_576167", "turns": [{"role": "user", "content": "How do the types of lenses used in photography affect the final image captured?"}, {"role": "assistant", "content": "The types of lenses used in photography affect the final image in several ways: 1. Focal length: The focal length of a lens determines its angle of view and the magnification of the subject. A longer focal length lens (telephoto) will compress the perspective and create a shallower depth of field, while a shorter focal length lens (wide-angle) will exaggerate the perspective and have a wider depth of field. 2."}, {"role": "user", "content": "How can I choose the right lens for a particular subject or situation in photography?"}, {"role": "assistant", "content": "Choosing the right lens for a particular subject or situation in photography can greatly impact the final image. Here are some considerations to help you choose the right lens: 1. Focal length: Consider the distance between you and the subject. If you are shooting a subject from afar, then consider using a longer focal length lens (telephoto). If you are closer to your subject, then consider using a shorter focal length lens (wide-angle). 2. Aperture: Consider the lighting conditions."}]}
{"filler_id": "FL28", "session_id": "ultrachat_182518", "turns": [{"role": "user", "content": "How has the mining industry impacted the economy and social structure of San Luis Potosí throughout history?"}, {"role": "assistant", "content": "The mining industry has had a significant impact on the economy and social structure of San Luis Potosí throughout history. The discovery of silver deposits in the region in the 16th century led to the establishment of numerous mines and mining towns, which in turn attracted a large population of migrants seeking work and other opportunities."}, {"role": "user", "content": "It's really interesting to learn about the impact of mining on San Luis Potosí. Have there been any efforts to address the negative consequences of the industry?"}, {"role": "assistant", "content": "Yes, there have been efforts to address the negative consequences of the mining industry in San Luis Potosí. In recent years, there has been increased awareness of the environmental impacts of mining on local communities and ecosystems, leading to the adoption of stricter regulations and the development of more sustainable mining practices."}]}
{"filler_id": "FL29", "session_id": "98cd882c_1", "turns": [{"role": "user", "content": "I'm looking for some ideas for a new sewing project. I just finished making a quilt for my sister's birthday and I'm itching to start something new. By the way, it took me around 3 weeks to complete the quilt, and I'm really happy with how it turned out."}, {"role": "assistant", "content": "Congratulations on completing the quilt for your sister's birthday! That's a wonderful accomplishment, and I'm sure she'll treasure it. Three weeks is a great timeframe to complete a quilt, considering the amount of time and effort that goes into it. Now, let's get started on brainstorming ideas for your next sewing project! Here are a few suggestions to get you going: 1. Tote Bag or Purse: A functional and practical project that can be completed quickly."}, {"role": "user", "content": "I've been thinking about making a sundress for summer, maybe something with a floral pattern. I've never made a garment before, so I'm a bit nervous about it. Do you have any tips or patterns that you would recommend for a beginner like me?"}, {"role": "assistant", "content": "Making a sundress is a great project for a beginner! With a little guidance, you'll be able to create a beautiful and comfortable garment for the summer. I'm happy to help you with tips and pattern recommendations. Tips for a Beginner: 1. Choose a simple pattern: Look for patterns labeled \"beginner-friendly\" or \"easy.\" Avoid patterns with too many pieces, complex construction, or intricate details. 2. Select a forgiving fabric: Opt for a lightweight, breathable fabric like cotton, linen, or rayon."}]}
{"filler_id": "FL30", "session_id": "f5f1ff92_1", "turns": [{"role": "user", "content": "I'm looking to get some advice on the mortgage application process. I've already submitted all the required documents to the lender, including my pay stubs, bank statements, and tax returns, but I'm not sure what to expect next. Can you walk me through the typical steps after submitting the application?"}, {"role": "assistant", "content": "Congratulations on submitting your mortgage application! After submitting your documents, the lender will review and process your application. Here's a general outline of the typical steps that follow: 1. Initial Review: The lender's loan processor will review your application to ensure all required documents are complete and in order. This step usually takes 1-3 business days. 2. Loan Processing: The loan processor will review your credit report, verify your income, and evaluate your debt-to-income ratio."}, {"role": "user", "content": "I'm a bit concerned about the appraisal step. Can you tell me more about what happens during an appraisal and how it affects the loan approval process?"}, {"role": "assistant", "content": "The appraisal step is a crucial part of the mortgage process, and it's natural to have questions about it. Here's a detailed explanation of what happens during an appraisal and how it affects the loan approval process: What is an appraisal? A real estate appraisal is an independent evaluation of a property's value, conducted by a licensed appraiser. The appraiser provides an objective opinion of the property's market value, based on various factors such as: 1."}]}
{"filler_id": "FL31", "session_id": "39948bcf_4", "turns": [{"role": "user", "content": "I'm looking for some recommendations on yoga classes in my area. I've been interested in continuing my practice since I attended a weekend-long yoga retreat last year in November, which included several workshops on meditation, pranayama, and asana practice."}, {"role": "assistant", "content": "Wonderful! It's great to hear that you're inspired to continue your yoga practice after the retreat. Here are some tips to help you find a yoga class that suits your needs: 1. Online Search: Type in \"yoga classes near me\" or \"yoga studios in [your city/zip code]\" in a search engine like Google. You can also use online directories like YogaFinder or Yoga Alliance to find studios and classes in your area. 2."}, {"role": "user", "content": "I'm actually interested in finding a studio that focuses on restorative yoga, as I really enjoyed that particular workshop during the retreat. Do you know of any studios in my area that specialize in restorative yoga?"}, {"role": "assistant", "content": "Wonderful! Restorative yoga is a great focus for a studio, and it's excellent that you're looking to deepen your practice in that area. Unfortunately, I'm a large language model, I don't have access to your location or real-time information about specific studios in your area. However, I can guide you on how to find a restorative yoga studio near you: 1."}]}
{"filler_id": "FL32", "session_id": "ultrachat_82739", "turns": [{"role": "user", "content": "Are backstage passes usually limited to a certain number of people? If so, how can I increase my chances of securing one?"}, {"role": "assistant", "content": "Yes, backstage passes are usually limited to a certain number of people. To increase your chances of securing one, here are some tips: 1. Join fan clubs or mailing lists: Many artists and bands offer exclusive access to backstage passes or pre-sales of tickets to their fan clubs or mailing lists. 2. Follow them on social media: Artists and bands often announce giveaways or promotions for backstage passes on their social media platforms. 3."}, {"role": "user", "content": "I'll definitely start following them on social media and enter some contests. Do you know if there are any upcoming concerts with backstage passes available?"}, {"role": "assistant", "content": "As an AI language model, I do not have access to real-time information about upcoming concerts with backstage passes available. However, you can check the artist or band's official website or social media pages for information about their upcoming concerts and any backstage pass opportunities they may have available. You can also check with the concert venue or ticketing agency for any special VIP or backstage packages they offer."}]}
{"filler_id": "FL33", "session_id": "f340a3db", "turns": [{"role": "user", "content": "I'm planning to launch a social media campaign on May 1st to raise awareness about mental health, can you help me come up with some engaging post ideas and hashtags?"}, {"role": "assistant", "content": "What a wonderful initiative! I'd be happy to help you with engaging post ideas and hashtags for your social media campaign to raise awareness about mental health. Here are some suggestions: Post Ideas: 1. Personal Stories: Share personal stories of struggle and triumph over mental health issues. Encourage your followers to share their own stories using a branded hashtag. 2."}, {"role": "user", "content": "Can you suggest some popular mental health advocates or influencers I can reach out to for collaborations or Q&A sessions?"}, {"role": "assistant", "content": "Here are some popular mental health advocates and influencers across various platforms who you may consider reaching out to for collaborations or Q&A sessions: Mental Health Advocates: 1. Lady Gaga (@ladygaga) - Singer-songwriter and founder of the Born This Way Foundation, which focuses on mental health and wellness. 2. Demi Lovato (@ddlovato) - Singer-songwriter and mental health advocate who has spoken openly about her struggles with depression and addiction. 3."}]}
{"filler_id": "FL34", "session_id": "ultrachat_245735", "turns": [{"role": "user", "content": "What cultural barriers have hindered the preservation of the Kiowa language, and how have they been overcome?"}, {"role": "assistant", "content": "Several cultural barriers have hindered the preservation of the Kiowa language, including historical trauma, forced assimilation, and the acculturation of younger generations. In the late 1800s, the U.S. government forced indigenous peoples onto reservations and established boarding schools where they were not allowed to speak their native languages. This caused Kiowa children to be removed from their families and communities, leading to intergenerational trauma and a disconnection from their language and culture."}, {"role": "user", "content": "That's really interesting. Do you know how successful these language revitalization programs have been so far?"}, {"role": "assistant", "content": "Yes, there have been some successes in Kiowa language revitalization efforts. For example, the Kiowa Tribe's language immersion program in Anadarko, Oklahoma, has seen positive results in terms of language acquisition and usage. The program focuses on teaching the language to young children and providing cultural exposure to help them develop a strong sense of identity and connection to their community."}]}
{"filler_id": "FL35", "session_id": "ultrachat_20447", "turns": [{"role": "user", "content": "How has remote work affected the social and emotional wellbeing of employees?"}, {"role": "assistant", "content": "As an AI language model, I do not have personal experiences, opinions, or emotions. However, studies and research have shown that remote work can have both positive and negative impacts on the social and emotional wellbeing of employees. Positive impacts of remote work on social and emotional wellbeing: 1. Flexible work arrangements can reduce stress and burnout associated with long commutes and demanding schedules. 2."}, {"role": "user", "content": "Can employers implement any strategies to prevent remote workers from feeling lonely and isolated while working from home?"}, {"role": "assistant", "content": "Yes, employers can implement various strategies to help remote workers feel less lonely and isolated while working from home. Here are a few strategies: 1. Encourage regular virtual meetings to keep remote workers connected. Schedule regular video check-ins and team calls to foster a sense of belonging and collaboration. 2. Create virtual coffee breaks, lunches or happy hours. These breaks help employees to take a break, socialize with colleagues and rejuvenate. 3. Facilitate team building activities."}]}
{"filler_id": "FL36", "session_id": "9f35ac68", "turns": [{"role": "user", "content": "I'm thinking of switching to a monthly bus pass, can you tell me the cost and where I can buy one?"}, {"role": "assistant", "content": "I'd be happy to help! However, I need a bit more information from you. Could you please provide me with the following details: 1. The city or region you live in (or plan to use the bus pass in)? 2. The type of bus pass you're interested in (e.g. adult, student, senior, disabled)? 3. The duration of the pass you're looking for (e.g. 1 month, 3 months, 12 months)?"}, {"role": "user", "content": "I live in the city, and I'm interested in an adult monthly pass. I'm looking for a 1-month pass."}, {"role": "assistant", "content": "I'd be happy to help! However, I need to clarify that I'm a large language model, I don't have access to real-time information about specific cities or transportation systems. But I can give you some general guidance on how to find the information you're looking for. To get the exact cost and details on where to buy a monthly bus pass in your city, I recommend the following steps: 1."}]}
{"filler_id": "FL37", "session_id": "285a29e7_2", "turns": [{"role": "user", "content": "I'm looking for some book recommendations. I've been reading a lot lately, and I'm always up for discovering new authors. By the way, I attended a poetry reading at a local coffee shop today, where several local poets read from their recent collections - it was amazing to see so many talented people share their work. Anyway, what are some popular books or authors you'd suggest?"}, {"role": "assistant", "content": "That sounds like a wonderful experience! Attending a poetry reading at a local coffee shop is a great way to discover new talent and connect with the literary community. As for book recommendations, I'd be happy to suggest some popular books and authors across various genres. Keep in mind that personal tastes may vary, but these are some widely acclaimed and popular choices: Fiction: 1."}, {"role": "user", "content": "I'd love to explore some poetry recommendations more. The poetry reading I attended today really opened my eyes to the talent out there. Do you have any suggestions for contemporary poetry collections or poets that I might enjoy, given my interest in discovering new voices?"}, {"role": "assistant", "content": "I'm thrilled to hear that the poetry reading inspired you to explore more poetry! Contemporary poetry is a vibrant and diverse field, and there are many incredible poets and collections worth discovering. Here are some recommendations that might resonate with you: Contemporary Poetry Collections: 1. When I Grow Up I Want to Be a List of Further Possibilities by Chen Chen: A debut collection that explores identity, family, love, and the search for meaning in a chaotic world. 2."}]}
{"filler_id": "FL38", "session_id": "sharegpt_NRqjZJS_0", "turns": [{"role": "user", "content": "Write an essay comparing and contrasting Ernest Hemmingway and John Steinbeck's writing styles"}, {"role": "assistant", "content": "Ernest Hemingway and John Steinbeck are two of the most renowned authors of the 20th century. Both men wrote about the struggles of the common man and the human condition, but their writing styles were vastly different. Hemingway is known for his concise, direct, and simple writing style. He uses short sentences and a minimalist approach to convey his message."}, {"role": "user", "content": "Please include specific examples by referencing some of the books they have written"}, {"role": "assistant", "content": "Ernest Hemingway and John Steinbeck are two of the most renowned authors of the 20th century. Both men wrote about the struggles of the common man and the human condition, but their writing styles were vastly different. Hemingway is known for his concise, direct, and simple writing style."}]}
{"filler_id": "FL39", "session_id": "0daaefa7_3", "turns": [{"role": "user", "content": "I'm trying to get into web development, and I was wondering if you could recommend some online resources for learning JavaScript. By the way, I just attended a 2-day coding bootcamp in mid-February, where I learned the basics of JavaScript, but I want to continue improving my skills."}, {"role": "assistant", "content": "Congratulations on taking the first step into web development and attending a coding bootcamp! With a solid foundation in JavaScript basics, you're ready to dive deeper and improve your skills. Here are some online resources to help you continue learning JavaScript: Tutorials and Courses: 1. Codecademy's JavaScript Course: A comprehensive and interactive course that covers JavaScript fundamentals, intermediate topics, and advanced concepts. 2. FreeCodeCamp: A non-profit platform that offers a range of JavaScript challenges, projects, and certifications."}, {"role": "user", "content": "I'll definitely check out these resources. I'm especially interested in practicing with JSFiddle and HackerRank's challenges. Do you have any tips on how to stay motivated and avoid burnout while learning to code?"}, {"role": "assistant", "content": "Practicing with JSFiddle and HackerRank's challenges is an excellent way to improve your JavaScript skills. Staying motivated and avoiding burnout are crucial aspects of learning to code. Here are some tips to help you: 1. Set achievable goals and milestones: Break down your learning journey into smaller, manageable goals. Celebrate your achievements, no matter how small they may seem. This will help you stay motivated and track your progress. 2."}]}
{"filler_id": "FL40", "session_id": "ultrachat_134317", "turns": [{"role": "user", "content": "How did the Emperor's role as head of the religion of the Roman state contribute to their authority?"}, {"role": "assistant", "content": "The Emperor's role as head of the religion of the Roman state was significant in contributing to their authority because religion played a significant role in the lives of the Roman people. The Emperors had control over religious institutions, rituals, and ceremonies, which allowed them to present themselves as the guardians of the traditional Roman values and customs."}, {"role": "user", "content": "Interesting, so did the Emperors have to follow certain religious practices or was their position more of a symbolic one?"}, {"role": "assistant", "content": "The Emperor's position as head of the religion of the Roman state was not just symbolic; rather, they were expected to follow certain religious practices and participate in religious ceremonies. The Emperor's religious duties included performing sacrifices, offering prayers, and officiating at important religious ceremonies. The Emperor was believed to have a special relationship with the gods and was regarded as the high priest of the Roman religion."}]}


## 3b. Stage runner (stops Run All if a stage fails)

In [ ]:
def stage(name, extra=""):
    ip = get_ipython()
    ip.system(f"cd /kaggle/working/clm/src && CLM_ROOT=/kaggle/working/clm CLM_RUN=batch01_pilot_r2 CLM_BACKEND={os.environ['CLM_BACKEND']} "
              f"python -m clm.stages {name} --items /kaggle/working/clm/data/items/items_batch01_pilot.jsonl {extra}")
    if ip.user_ns.get("_exit_code", 0) != 0:
        raise RuntimeError(f"Stage '{name}' failed - copy the error above to Claude.")
    print(f"stage {name}: done")

## 4. Translate fillers, build histories, translate memories/queries to English

In [ ]:
stage("translate")

## 5. Retrieval

In [ ]:
stage("retrieve")

## 6. Token-length check (Hindi/Hinglish need more tokens)

In [ ]:
stage("tokens")

## 7. Reader: Qwen3-8B (then free disk)

In [ ]:
stage("generate", "--model qwen3-8b")
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3-8B

## 8. Reader: Llama-3.1-8B-Instruct (then free disk)

In [ ]:
stage("generate", "--model llama3.1-8b")
!rm -rf ~/.cache/huggingface/hub/models--meta-llama--Llama-3.1-8B-Instruct

## 9. Judge: Qwen3-14B-AWQ

In [ ]:
stage("judge")

## 10. Report and download

In [ ]:
stage("report")
import shutil
shutil.make_archive("/kaggle/working/results_batch01_pilot_r2", "zip", "/kaggle/working/clm/results/batch01_pilot_r2")
print("Download: /kaggle/working/results_batch01_pilot_r2.zip  (right panel -> Output)")